# Pipeline Results Visualization Notebook

This notebook is an interactive refactor of the pipeline plotting script.

It is organized by pipeline stage so you can run, inspect, and edit each figure as you iterate:
1. Baseline flow outputs
2. Flood disruption outputs
3. Direct damage outputs
4. Rerouting and recovery outputs

Optional hazard map code is included at the end for future hazard layers.

In [1]:
from __future__ import annotations

import ast
import os
from pathlib import Path

import geopandas as gpd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import contextily as ctx
    HAS_CONTEXTILY = True
except ImportError:
    HAS_CONTEXTILY = False

plt.style.use("default")

In [2]:
# Parameters to update for each run
# Override any of these with environment variables when needed:

#   NIRD_RESULTS_ROOT   - folder containing base_scenario/disruption_analysis/etc.
#   NIRD_RESULTS_VARIANT - variant subfolder name (for example: revision, od_inflation_100x)
#   NIRD_INPUT_ROOT     - folder containing parameters, damage_curves, study_area, etc.
#   NIRD_DEPTH_KEY      - flood depth threshold folder to read (for example: 15, 30, 60)
#   NIRD_FLOOD_KEY      - flood scenario ID (for example: 1, 2, 3)

def _first_existing(paths):
    for path in paths:
        if path and Path(path).exists():
            return Path(path)
    return None


root = Path.cwd()
while root != root.parent and not (root / "pyproject.toml").exists():
    root = root.parent

home = Path.home()
results_candidates = []
if os.getenv("NIRD_RESULTS_ROOT"):
    results_candidates.append(Path(os.getenv("NIRD_RESULTS_ROOT")))
results_candidates.extend([
    home / "NIRD_Data" / "results",
    root / "results",
    root / "sandbox" / "results",
])
results_root = _first_existing(results_candidates)
if results_root is None:
    raise FileNotFoundError("Could not locate a results root. Set NIRD_RESULTS_ROOT.")

input_candidates = []
if os.getenv("NIRD_INPUT_ROOT"):
    input_candidates.append(Path(os.getenv("NIRD_INPUT_ROOT")))
input_candidates.extend([
    home / "NIRD_Data" / "va_soge_clusters_toy",
    home / "NIRD_Data" / "fairfax_soge_clusters_toy",
    root / "sandbox" / "fairfax_soge_clusters_toy",
    root / "sandbox" / "fairfax_baseline",
])
input_root = _first_existing(input_candidates)
if input_root is None:
    raise FileNotFoundError("Could not locate an input root. Set NIRD_INPUT_ROOT.")

variant_env = os.getenv("NIRD_RESULTS_VARIANT")
variant_candidates = sorted([p.name for p in (results_root / "base_scenario").glob("*") if p.is_dir()]) if (results_root / "base_scenario").exists() else []
if variant_env and (results_root / "base_scenario" / variant_env).exists():
    VARIANT = variant_env
elif "revision" in variant_candidates:
    VARIANT = "revision"
elif "od_inflation_100x" in variant_candidates:
    VARIANT = "od_inflation_100x"
elif len(variant_candidates) == 1:
    VARIANT = variant_candidates[0]
elif variant_candidates:
    VARIANT = variant_candidates[0]
else:
    VARIANT = variant_env or "revision"

def _available_depth_keys(variant: str) -> list[int]:
    disruption_root = results_root / "disruption_analysis" / variant
    if not disruption_root.exists():
        return []
    depth_keys = []
    for p in disruption_root.iterdir():
        if p.is_dir() and p.name.isdigit():
            depth_keys.append(int(p.name))
    return sorted(set(depth_keys))

depth_env = os.getenv("NIRD_DEPTH_KEY")
depth_candidates = _available_depth_keys(VARIANT)
if depth_env is not None and str(depth_env).isdigit():
    DEPTH_KEY = int(depth_env)
elif 15 in depth_candidates:
    DEPTH_KEY = 15
elif 30 in depth_candidates:
    DEPTH_KEY = 30
elif depth_candidates:
    DEPTH_KEY = depth_candidates[0]
else:
    DEPTH_KEY = 30

def _available_flood_ids(variant: str, depth_key: int) -> list[int]:
    links_dir = results_root / "disruption_analysis" / variant / str(depth_key) / "links"
    if not links_dir.exists():
        return []
    flood_ids = []
    for p in links_dir.glob("road_links_*.gpq"):
        tail = p.stem.split("_")[-1]
        if tail.isdigit():
            flood_ids.append(int(tail))
    return sorted(set(flood_ids))

flood_env = os.getenv("NIRD_FLOOD_KEY")
flood_candidates = _available_flood_ids(VARIANT, DEPTH_KEY)
if flood_env is not None and str(flood_env).isdigit():
    FLOOD_KEY = int(flood_env)
elif 1 in flood_candidates:
    FLOOD_KEY = 1
elif flood_candidates:
    FLOOD_KEY = flood_candidates[0]
else:
    FLOOD_KEY = 1

STUDY_AREA_LABEL = os.getenv("NIRD_STUDY_AREA_LABEL", "Virginia")
HAZARD_TYPE_LABEL = os.getenv("NIRD_HAZARD_TYPE_LABEL", "Flood")
SCENARIO_SPECS = [("Baseline", 1, "#b279a2"), ("Low", 2, "#4c78a8"), ("High", 3, "#f58518")]
SCENARIO_EVENT_SPECS = [("baseline", 1), ("low", 2), ("high", 3)]

RECOVERY_DAYS = [0, 7, 14, 30, 60, 90, 120, 150, 180]

results_variant_root = results_root / "base_scenario" / VARIANT
fig_root = results_root / "figures" / VARIANT

step1_fig_dir = fig_root / "step1_base"
step2_fig_dir = fig_root / "step2_disruption"
step3_fig_dir = fig_root / "step3_damage"
step4_fig_dir = fig_root / "step4_rerouting"
for p in [step1_fig_dir, step2_fig_dir, step3_fig_dir, step4_fig_dir]:
    p.mkdir(parents=True, exist_ok=True)

import sys
viz_dir = root / "scripts" / "visualizations"
if str(viz_dir) not in sys.path:
    sys.path.insert(0, str(viz_dir))
from viz_data_loaders import (
    build_flow_validation_table,
    build_scenario_summary_table,
    event_damage_total_usd,
    format_cost,
    format_usd_millions,
    is_testbed_variant,
    list_available_flood_keys,
    load_assignment_od_demand,
    load_edge_flows,
    load_odpfc,
    load_sctg_summary,
    prepare_damage_for_viz,
    resolve_edge_flows_path,
    resolve_odpfc_path,
    should_skip_map_layers,
    subset_links_for_map,
)

paths = {
    "base_odpfc": results_variant_root / "odpfc.pq",
    "base_odpfc_parts": results_variant_root / "odpfc_parts",
    "base_edge_flows": results_variant_root / "edge_flows.gpq",
    "base_edge_flows_pass_a": results_variant_root / "edge_flows_pass_a.gpq",
    "disruption_intersections": results_root / "disruption_analysis" / VARIANT / str(DEPTH_KEY) / "intersections" / f"intersections_{FLOOD_KEY}.pq",
    "disruption_links": results_root / "disruption_analysis" / VARIANT / str(DEPTH_KEY) / "links" / f"road_links_{FLOOD_KEY}.gpq",
    "damage_csv": results_root / "damage_analysis" / VARIANT / f"intersections_{FLOOD_KEY}_with_damage_values.csv",
    "rerouting_costs": results_root / "rerouting_analysis" / VARIANT / str(DEPTH_KEY) / str(FLOOD_KEY) / "cost_matrix_by_scenario.csv",
    "rerouting_dir": results_root / "rerouting_analysis" / VARIANT / str(DEPTH_KEY) / str(FLOOD_KEY),
}

RUN_CONTEXT = {
    "study_area_label": STUDY_AREA_LABEL,
    "hazard_type_label": HAZARD_TYPE_LABEL,
    "baseline_scenario_label": "Baseline",
    "scenario_set_label": "Baseline / Low / High",
    "selected_scenario_label": f"Selected flood event (id={FLOOD_KEY})",
    "scenario_specs": SCENARIO_SPECS,
    "scenario_event_specs": SCENARIO_EVENT_SPECS,
}

def _infer_scenario_label(title: str) -> str:
    lower = title.lower()
    if "baseline vs" in lower or "by scenario" in lower or "scenario comparison" in lower or "all links" in lower or "across scenarios" in lower:
        return RUN_CONTEXT["scenario_set_label"]
    if "baseline" in lower:
        return RUN_CONTEXT["baseline_scenario_label"]
    if "input:" in lower:
        return "Reference input"
    return RUN_CONTEXT["selected_scenario_label"]

# Safe, idempotent monkey-patch for contextual suptitle
import importlib, matplotlib.figure as _mpl_fig

# reload the figure module to get a fresh original suptitle implementation (clears runtime wrappers)
_mpl_fig = importlib.reload(_mpl_fig)
_mpl_original = _mpl_fig.Figure.suptitle

# persist the true original on the class so our wrapper can call it
plt.Figure._nird_original_suptitle = _mpl_original

def _contextual_suptitle(self, t, *args, **kwargs):
    original = getattr(plt.Figure, "_nird_original_suptitle")
    scenario_override = getattr(self, "_nird_plot_scenario", None)
    if isinstance(t, str) and "Study area:" not in t and "Hazard type:" not in t and "Scenario:" not in t:
        scenario_label = scenario_override or _infer_scenario_label(t)
        t = (
            f"{t} | Study area: {RUN_CONTEXT['study_area_label']}"
            f" | Hazard type: {RUN_CONTEXT['hazard_type_label']}"
            f" | Scenario: {scenario_label}"
        )
    return _mpl_original(self, t, *args, **kwargs)

# replace the public attribute with our wrapper (idempotent assignment)
plt.Figure.suptitle = _contextual_suptitle

print(f"Using results_root: {results_root}")
print(f"Using input_root: {input_root}")
print(f"Using VARIANT: {VARIANT}")
print(f"Using DEPTH_KEY: {DEPTH_KEY}")
print(f"Using FLOOD_KEY: {FLOOD_KEY}")

resolved_odpfc = resolve_odpfc_path(results_variant_root)
resolved_edge_flows = resolve_edge_flows_path(results_variant_root)
print(f"Resolved odpfc source: {resolved_odpfc}")
print(f"Resolved edge-flow source: {resolved_edge_flows}")

for k, v in paths.items():
    print(f"{k}: {v}")
    if k != "rerouting_dir":
        print(f"  exists: {v.exists()}")


Using results_root: C:\Users\akothaw\Desktop\data\results
Using input_root: C:\Users\akothaw\Desktop\data\soge_clusters
Using VARIANT: revision
Using DEPTH_KEY: 30
Using FLOOD_KEY: 1
Resolved odpfc source: C:\Users\akothaw\Desktop\data\results\base_scenario\revision\odpfc.pq
Resolved edge-flow source: C:\Users\akothaw\Desktop\data\results\base_scenario\revision\edge_flows_pass_a.gpq
base_odpfc: C:\Users\akothaw\Desktop\data\results\base_scenario\revision\odpfc.pq
  exists: True
base_odpfc_parts: C:\Users\akothaw\Desktop\data\results\base_scenario\revision\odpfc_parts
  exists: True
base_edge_flows: C:\Users\akothaw\Desktop\data\results\base_scenario\revision\edge_flows.gpq
  exists: True
base_edge_flows_pass_a: C:\Users\akothaw\Desktop\data\results\base_scenario\revision\edge_flows_pass_a.gpq
  exists: True
disruption_intersections: C:\Users\akothaw\Desktop\data\results\disruption_analysis\revision\30\intersections\intersections_1.pq
  exists: True
disruption_links: C:\Users\akothaw\De

## Scenario QA summary

Headline metrics for the selected variant and depth. Cost columns use **KUSD** on
testbed variants (`toy_*`, `*_sioux_falls`, or `NIRD_TESTBED=1`) and **MUSD** on
production-scale runs unless `NIRD_COST_DISPLAY_UNIT` overrides auto selection.

In [ ]:
flood_keys = list_available_flood_keys(results_root, VARIANT, DEPTH_KEY) or [FLOOD_KEY]
scenario_summary = build_scenario_summary_table(
    results_root,
    VARIANT,
    DEPTH_KEY,
    flood_keys=flood_keys,
)
if scenario_summary.empty:
    print("No scenario summary rows found for the current variant/depth.")
else:
    unit_hint = "KUSD (testbed)" if is_testbed_variant(VARIANT) else "MUSD (production-scale)"
    print(f"Scenario QA | variant={VARIANT} | depth={DEPTH_KEY} | units={unit_hint}")
    display_cols = [
        "flood_key",
        "flooded_links",
        "closed_links",
        "passenger_disrupted_flow",
        "freight_disrupted_flow",
        "rerouting_cost_passenger_display",
        "rerouting_cost_freight_display",
        "direct_damage_display",
        "combined_total_display",
        "isolation_rows",
    ]
    present = [c for c in display_cols if c in scenario_summary.columns]
    display(scenario_summary[present])

In [3]:
def save_figure(fig: plt.Figure, out_dir: Path, stem: str) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_dir / f"{stem}.png", dpi=300, bbox_inches="tight", facecolor="white")
    fig.savefig(out_dir / f"{stem}.pdf", bbox_inches="tight", facecolor="white")

def parse_path_edges(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, (list, tuple, set)):
        return [str(v) for v in value]
    if isinstance(value, np.ndarray):
        values = value.tolist()
        if values and all(isinstance(v, str) and len(v) == 1 for v in values):
            s = "".join(values)
            if s.startswith("[") and s.endswith("]"):
                try:
                    parsed = ast.literal_eval(s)
                    if isinstance(parsed, (list, tuple, set, np.ndarray)):
                        return [str(v) for v in parsed]
                except Exception:
                    pass
        return [str(v) for v in values]
    if isinstance(value, str):
        s = value.strip()
        if s.startswith("[") and s.endswith("]"):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set, np.ndarray)):
                    return [str(v) for v in parsed]
            except Exception:
                pass
        return [tok.strip() for tok in s.split(",") if tok.strip()]
    return [str(value)]

def compute_path_distance_miles(odpfc_df: pd.DataFrame, edges_gdf: gpd.GeoDataFrame) -> pd.Series:
    edge_len_col = "length_mile" if "length_mile" in edges_gdf.columns else "length"
    edge_lengths = edges_gdf.assign(e_id=edges_gdf["e_id"].astype(str)).set_index("e_id")[edge_len_col].astype(float)
    if edge_len_col == "length":
        edge_lengths = edge_lengths / 1609.34

    def _dist(path_val):
        edges = parse_path_edges(path_val)
        return float(sum(edge_lengths.get(str(e), 0.0) for e in edges))

    return odpfc_df["path"].apply(_dist)

In [4]:
# Load all data once
assignment_od, assignment_od_source = load_assignment_od_demand(input_root)
print(f"Loaded full assignment OD from: {assignment_od_source}")
try:
    odpfc, odpfc_source = load_odpfc(results_variant_root)
    print(f"Loaded assigned paths from: {odpfc_source}")
except FileNotFoundError:
    odpfc = None
    odpfc_source = None
    print("Assigned OD paths (odpfc) not found; distance-vs-flow plot uses demand only.")
edge_flows, edge_flows_source = load_edge_flows(results_variant_root)
print(f"Loaded edge flows from: {edge_flows_source}")
intersections = pd.read_parquet(paths["disruption_intersections"])
road_links = gpd.read_parquet(paths["disruption_links"])
map_edge_flows = subset_links_for_map(edge_flows)
map_road_links = subset_links_for_map(road_links, flow_col="current_flow")
skip_map_layers = should_skip_map_layers(edge_flows)
damage_df = prepare_damage_for_viz(pd.read_csv(paths["damage_csv"]))

rerouting_costs_path = paths["rerouting_costs"]
if rerouting_costs_path.exists():
    rerouting_costs = pd.read_csv(rerouting_costs_path).sort_values("scenario")
else:
    rerouting_costs = pd.DataFrame()

print("Loaded shapes:")
print("  assignment_od:", assignment_od.shape)
print("  odpfc:", None if odpfc is None else odpfc.shape)
print("  edge_flows:", edge_flows.shape)
print("  intersections:", intersections.shape)
print("  road_links:", road_links.shape)
print("  damage_df:", damage_df.shape)
print("  rerouting_costs:", rerouting_costs.shape)
print(
    "  direct damage (consolidated USD):",
    format_usd_millions(event_damage_total_usd(damage_df)),
)


Loaded odpfc from: C:\Users\akothaw\Desktop\data\results\base_scenario\revision\odpfc.pq
Loaded edge flows from: C:\Users\akothaw\Desktop\data\results\base_scenario\revision\edge_flows_pass_a.gpq
Loaded shapes:
  odpfc: (19351, 8)
  edge_flows: (483442, 35)
  intersections: (1268351, 8)
  road_links: (483599, 28)
  damage_df: (770, 68)
  rerouting_costs: (1, 7)


## Step 1. Baseline flow visuals

### Baseline flow analysis (4 plots)
- **Plot 1 (top-left)**: Distribution of OD flow volumes in vehicles per day (raw daily volumes, no scaling)
- **Plot 2 (top-right)**: Relationship between route distance and flow, with fitted log-linear trend (longer routes typically carry less flow)
- **Plot 3 (bottom-left)**: The 12 largest origin-destination pairs by volume
- **Plot 4 (bottom-right)**: Spatial view of baseline network flows, where line thickness scales with accumulated flow

In [5]:
flow_col = "flow" if "flow" in assignment_od.columns else "Car21"
od_demand_step1 = assignment_od.copy()
total_flow = float(od_demand_step1[flow_col].sum())
top_share = float(od_demand_step1[flow_col].nlargest(min(10, len(od_demand_step1))).sum() / total_flow) if total_flow > 0 else np.nan
valid = pd.DataFrame()
path_flow_col = flow_col
if odpfc is not None and "path" in odpfc.columns:
    odpfc_step1 = odpfc.copy()
    odpfc_step1["distance_miles"] = compute_path_distance_miles(odpfc_step1, edge_flows)
    path_flow_col = "flow" if "flow" in odpfc_step1.columns else "Car21"
    valid = odpfc_step1[(odpfc_step1[path_flow_col] > 0) & (odpfc_step1["distance_miles"] > 0)].copy()

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

axes[0].hist(od_demand_step1[flow_col].astype(float), bins=30, color="#4c78a8", edgecolor="white")
axes[0].set_title(f"Full assignment OD demand ({len(od_demand_step1):,} pairs)")
axes[0].set_xlabel("Daily truck trips (FAF5 demand)")
axes[0].set_ylabel("Count")
axes[0].grid(True, alpha=0.2)

if len(valid) > 4:
    axes[1].scatter(valid["distance_miles"], valid[path_flow_col], s=12, alpha=0.35, color="#f58518")
    slope, intercept = np.polyfit(valid["distance_miles"].astype(float), np.log1p(valid[path_flow_col].astype(float)), 1)
    x = np.linspace(valid["distance_miles"].min(), valid["distance_miles"].max(), 100)
    y = np.expm1(intercept + slope * x)
    axes[1].plot(x, y, color="black", linewidth=1.5, label="log-linear fit")
    axes[1].legend(frameon=False, fontsize=8)
    axes[1].set_title("Flow versus path distance")
    axes[1].set_xlabel("Distance (miles)")
    axes[1].set_ylabel("Daily truck trips")
    axes[1].grid(True, alpha=0.2)
else:
    axes[1].text(0.5, 0.5, "Not enough valid rows for distance plot", ha="center", va="center")
    axes[1].set_axis_off()

top_od = od_demand_step1.sort_values(flow_col, ascending=False).head(12).copy()
top_od["od_pair"] = top_od["origin_node"].astype(str) + " -> " + top_od["destination_node"].astype(str)
axes[2].barh(top_od["od_pair"].iloc[::-1], top_od[flow_col].iloc[::-1], color="#54a24b")
axes[2].set_title("Largest OD flows")
axes[2].set_xlabel("Daily truck trips")
axes[2].grid(True, axis="x", alpha=0.2)

if skip_map_layers:
    axes[3].text(
        0.5,
        0.5,
        f"Network map omitted for large study area\n({len(edge_flows):,} links)",
        ha="center",
        va="center",
        fontsize=10,
    )
    axes[3].set_axis_off()
else:
    map_df = map_edge_flows.copy()
    map_df["acc_flow"] = pd.to_numeric(map_df["acc_flow"], errors="coerce").fillna(0.0)
    map_df["linewidth"] = 0.1 + 2.4 * (map_df["acc_flow"] / max(map_df["acc_flow"].max(), 1.0))
    if map_df.crs:
        map_df = map_df.to_crs("EPSG:4326")
    map_df.plot(ax=axes[3], linewidth=map_df["linewidth"], color="#1f77b4", alpha=0.6)
    axes[3].set_title("Baseline network flow map")
    axes[3].set_axis_off()

fig.suptitle(f"Baseline summary | total flow={total_flow:,.0f} | top 10 share={top_share:.1%}", fontsize=14)
fig.tight_layout()
save_figure(fig, step1_fig_dir, "step1_baseline_summary")
plt.show()

C:\Users\akothaw\AppData\Local\Temp\ipykernel_37756\816659286.py:63: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# FAF5 industry (SCTG G5) breakdown from county/regional freight data
try:
    sctg_summary, sctg_source = load_sctg_summary(input_root)
    fig_sctg, ax_sctg = plt.subplots(figsize=(10, 6))
    ax_sctg.barh(sctg_summary["industry"].iloc[::-1], sctg_summary["daily_truck_trips"].iloc[::-1], color="#4c78a8")
    ax_sctg.set_xlabel("Daily truck trips")
    ax_sctg.set_title(f"FAF5 freight by industry ({sctg_source.name})")
    ax_sctg.grid(True, axis="x", alpha=0.2)
    fig_sctg.tight_layout()
    save_figure(fig_sctg, step1_fig_dir, "faf5_sctg_industry_breakdown")
    plt.show()
except FileNotFoundError as exc:
    print(f"SCTG breakdown skipped: {exc}")


## Step 2. Flood disruption visuals

In [6]:
road_links_step2 = road_links.copy()
intersections_step2 = intersections.copy()

flood_col = "flood_depth_max" if "flood_depth_max" in road_links_step2.columns else None
if flood_col is None:
    flood_col = next((c for c in road_links_step2.columns if c.startswith("flood_depth")), None)
if flood_col is None:
    road_links_step2["flood_depth_value"] = np.nan
    flood_col = "flood_depth_value"

damage_level_dict = {
    "no": 0,
    "minor": 1,
    "moderate": 2,
    "extensive": 3,
    "severe": 4,
}
damage_level_order = [0, 1, 2, 3, 4]
damage_level_label_by_num = {v: k for k, v in damage_level_dict.items()}

damage_level_col = "damage_level_max" if "damage_level_max" in road_links_step2.columns else next((c for c in road_links_step2.columns if c.startswith("damage_level")), None)
road_links_step2[flood_col] = pd.to_numeric(road_links_step2[flood_col], errors="coerce")

# Keep this summary focused on the two comparable metrics across scales:
# 1) flood depth distribution, 2) damage level count + percentage
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Panel A: flood depth distribution
road_links_step2[flood_col].dropna().plot(kind="hist", bins=30, color="#9e77c9", edgecolor="white", ax=axes[0])
axes[0].set_title("Flood depth distribution on exposed links")
axes[0].set_xlabel("Flood depth (m)")
axes[0].set_ylabel("Count (links)")
axes[0].grid(True, alpha=0.2)

# Panel B: damage levels in fixed 0->4 order, including zeros
if damage_level_col and damage_level_col in road_links_step2.columns:
    damage_raw = road_links_step2[damage_level_col]
    damage_num = pd.to_numeric(damage_raw, errors="coerce")

    # If values are text labels, map them to 0..4.
    if damage_num.isna().all():
        damage_num = damage_raw.astype(str).str.strip().str.lower().map(damage_level_dict)

    counts = damage_num.value_counts(dropna=True).reindex(damage_level_order, fill_value=0).astype(int)
    denominator = int(counts.sum())
    pct = (counts / denominator * 100.0) if denominator > 0 else counts.astype(float)

    x_labels = [damage_level_label_by_num[i] for i in counts.index]
    bars = axes[1].bar(x_labels, counts.values, color="#e45756")
    axes[1].set_title("Damage level counts (with percentages)")
    axes[1].set_xlabel("Damage level")
    axes[1].set_ylabel("Count (links)")
    axes[1].grid(True, axis="y", alpha=0.2)

    ymax = max(counts.max(), 1)
    axes[1].set_ylim(0, ymax * 1.2)

    for bar, c, p in zip(bars, counts.values, pct.values):
        axes[1].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + ymax * 0.03,
            f"{int(c)}\n({p:.1f}%)",
            ha="center",
            va="bottom",
            fontsize=8,
        )
else:
    axes[1].text(0.5, 0.5, "No damage level column found", ha="center", va="center")
    axes[1].set_axis_off()

n_links = int(len(road_links_step2))
fig.suptitle(f"Flood disruption summary | links={n_links:,}", fontsize=14)
fig.tight_layout()
save_figure(fig, step2_fig_dir, "step2_disruption_summary")
plt.show()

C:\Users\akothaw\AppData\Local\Temp\ipykernel_37756\3323262022.py:75: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# Flood disruption summary across Baseline / Low / High (2x3 layout)
# Top row (A): flood depth distributions on exposed links for each scenario
# Bottom row (B): damage level counts for each scenario

fig, axes = plt.subplots(2, 3, figsize=(18, 12), sharey=False)
axes_top = axes[0]
axes_bot = axes[1]

# iterate over all axes by converting to lists
for ax in list(axes_top) + list(axes_bot):
    ax.set_facecolor("#ffffff")

for idx, (label, flood_id, color) in enumerate(SCENARIO_SPECS):
    # load links for scenario
    links_path = results_root / "disruption_analysis" / VARIANT / str(DEPTH_KEY) / "links" / f"road_links_{flood_id}.gpq"
    if not links_path.exists():
        axes_top[idx].text(0.5, 0.5, f"Missing links for {label}", ha="center", va="center")
        axes_top[idx].set_axis_off()
        axes_bot[idx].text(0.5, 0.5, f"Missing links for {label}", ha="center", va="center")
        axes_bot[idx].set_axis_off()
        continue

    links_gdf = gpd.read_parquet(links_path).copy()
    flood_col = "flood_depth_max" if "flood_depth_max" in links_gdf.columns else next((c for c in links_gdf.columns if c.startswith("flood_depth")), None)
    if flood_col is None:
        axes_top[idx].text(0.5, 0.5, f"No flood depth column for {label}", ha="center", va="center")
        axes_top[idx].set_axis_off()
    else:
        depths = pd.to_numeric(links_gdf[flood_col], errors="coerce").dropna()
        exposed = depths[depths > 0]
        if len(exposed) == 0:
            axes_top[idx].text(0.5, 0.5, f"No exposed links ({label})", ha="center", va="center")
            axes_top[idx].set_axis_off()
        else:
            axes_top[idx].hist(exposed, bins=30, color=color, edgecolor="white", alpha=0.9)
            axes_top[idx].set_title(f"{label} (id={flood_id})")
            axes_top[idx].set_xlabel("Flood depth (m)")
            axes_top[idx].grid(True, alpha=0.2)

    # bottom: damage level counts if present
    damage_col = next((c for c in links_gdf.columns if c.startswith("damage_level")), None)
    if damage_col is None:
        axes_bot[idx].text(0.5, 0.5, f"No damage level column", ha="center", va="center")
        axes_bot[idx].set_axis_off()
    else:
        dam = links_gdf[damage_col]
        # try numeric, else map labels
        dam_num = pd.to_numeric(dam, errors="coerce")
        if dam_num.isna().all():
            damage_map = {"no":0,"minor":1,"moderate":2,"extensive":3,"severe":4}
            dam_num = dam.astype(str).str.strip().str.lower().map(damage_map)
        counts = dam_num.value_counts(dropna=True).sort_index()
        # ensure 0..4
        all_idx = range(0,5)
        counts = pd.Series({i: int(counts.get(i,0)) for i in all_idx})
        labels_map = ["no","minor","moderate","extensive","severe"]
        bars = axes_bot[idx].bar(labels_map, counts.values, color=color)
        # add numeric labels above each bar
        maxy = max(counts.values) if len(counts.values) else 1
        for rect, c in zip(bars, counts.values):
            h = float(rect.get_height())
            axes_bot[idx].text(rect.get_x() + rect.get_width()/2, h + maxy * 0.02, f"{int(c)}", ha="center", va="bottom", fontsize=8)

        axes_bot[idx].set_title(f"{label} damage levels (links={len(links_gdf)})")
        axes_bot[idx].set_xlabel("Damage level")
        axes_bot[idx].grid(True, axis="y", alpha=0.2)

# Add left-side panel labels (A) and (B) next to vertical axis label
# Place labels aligned roughly with the vertical center of the top/bottom row
fig.text(0.005, 0.74, "(A)", fontsize=13, fontweight="bold", va="center")
fig.text(0.005, 0.34, "(B)", fontsize=13, fontweight="bold", va="center")

# shared y-label for left column; increase labelpad to make room for the (A)/(B)
axes_top[0].set_ylabel("Count (links)", labelpad=28)
axes_bot[0].set_ylabel("Count (links)", labelpad=28)

fig.suptitle(f"Flood disruption summary | Study area: {RUN_CONTEXT['study_area_label']} | Hazard: {RUN_CONTEXT['hazard_type_label']}", fontsize=14)
fig.tight_layout(rect=[0.06, 0.03, 1, 0.95])
save_figure(fig, step2_fig_dir, "step2_disruption_summary_2x3_scenarios")
plt.show()

C:\Users\akothaw\AppData\Local\Temp\ipykernel_37756\2231774856.py:80: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 3. Direct damage visuals

### Direct damage analysis (4 plots)
- **Plot 1 (top-left)**: Distribution of asset damage values showing the spread of direct damages across all affected links
- **Plot 2 (top-right)**: Total damage aggregated by road classification (to identify which road types bear the most damage)
- **Plot 3 (bottom-left)**: Spatial map highlighting the most damaged links on the network, with line thickness proportional to damage value
- **Plot 3a**: Spatial map of damage on all links, so you can see the full network pattern instead of only the top 50 links
- **Plot 4 (bottom-right)**: Distribution of damage fractions (normalized by asset value), showing percentage of asset value at risk
- **Plot 4a**: Damage fraction comparison across baseline, low, and high scenarios
- **Note**: centroid connectors are excluded from the damage totals and damage plots because they are routing artifacts, not real roadway assets

In [8]:
damage_step3 = damage_df.copy()
damage_step3["total_damage_value"] = pd.to_numeric(
    damage_step3["direct_damage_mean_usd"], errors="coerce"
).fillna(0.0)
damage_fraction_cols = [c for c in damage_step3.columns if c.endswith("_damage_fraction")]

damage_step3["total_damage_fraction"] = damage_step3[damage_fraction_cols].apply(pd.to_numeric, errors="coerce").max(axis=1)
damage_step3["e_id"] = damage_step3["e_id"].astype(str)

road_class_lookup = road_links[["e_id"]].copy()
road_class_lookup["e_id"] = road_class_lookup["e_id"].astype(str)
if "road_classification_detail" in road_links.columns:
    road_class_lookup["road_classification_detail"] = road_links["road_classification_detail"].astype(str)
elif "Class_Description" in road_links.columns:
    road_class_lookup["road_classification_detail"] = road_links["Class_Description"].astype(str)
else:
    road_class_lookup["road_classification_detail"] = road_links.get("road_classification", pd.Series(index=road_links.index, dtype=object)).astype(str)
road_class_lookup["road_classification_detail"] = road_class_lookup["road_classification_detail"].replace({"nan": np.nan, "None": np.nan, "": np.nan})
road_class_lookup = road_class_lookup.drop_duplicates(subset=["e_id"])
damage_step3 = damage_step3.merge(road_class_lookup, on="e_id", how="left")

centroid_mask = damage_step3["road_classification_detail"].astype(str).str.contains("centroid connector", case=False, na=False) | damage_step3["road_classification"].astype(str).str.contains("centroid connector", case=False, na=False)
centroid_damage_rows = int(centroid_mask.sum())
if centroid_damage_rows > 0:
    damage_step3 = damage_step3.loc[~centroid_mask].copy()

damage_step3["road_classification_detail"] = damage_step3["road_classification_detail"].fillna(damage_step3["road_classification"])

# Shared per-link aggregates for the separate plot cells below.
damage_per_link_value = damage_step3.groupby("e_id", as_index=False).agg(
    total_damage_value=("total_damage_value", "sum"),
)
damage_per_link_fraction = damage_step3.groupby("e_id", as_index=False).agg(
    total_damage_fraction=("total_damage_fraction", "max"),
)

if "road_classification_detail" in damage_step3.columns:
    by_class = damage_step3.groupby("road_classification_detail", dropna=False)["total_damage_value"].sum().sort_values(ascending=False)
elif "road_classification" in damage_step3.columns:
    by_class = damage_step3.groupby("road_classification", dropna=False)["total_damage_value"].sum().sort_values(ascending=False)
else:
    by_class = pd.Series(dtype=float)

print("Prepared damage summary inputs:")
print("  damage_step3:", damage_step3.shape)
print("  damage_per_link_value:", damage_per_link_value.shape)
print("  damage_per_link_fraction:", damage_per_link_fraction.shape)
print("  road_classification available:", not by_class.empty)
print("  centroid connectors excluded from damage totals:", centroid_damage_rows)

Prepared damage summary inputs:
  damage_step3: (770, 71)
  damage_per_link_value: (124, 2)
  damage_per_link_fraction: (124, 2)
  road_classification available: True
  centroid connectors excluded from damage totals: 0


In [9]:
# Plot 1: direct damage distributions for baseline / low / high scenarios
scenario_specs = SCENARIO_SPECS
scenario_damage = []
all_damage_values = []

for scenario_label, flood_id, color in scenario_specs:
    dmg_path = results_root / "damage_analysis" / VARIANT / f"intersections_{flood_id}_with_damage_values.csv"
    if not dmg_path.exists():
        print(f"Skipping {scenario_label}: missing {dmg_path}")
        continue

    dmg = prepare_damage_for_viz(pd.read_csv(dmg_path))
    damage_values = pd.to_numeric(dmg["direct_damage_mean_usd"], errors="coerce").dropna().astype(float)
    if len(damage_values) == 0:
        print(f"Skipping {scenario_label}: no consolidated damage values in {dmg_path.name}")
        continue

    scenario_damage.append({
        "label": scenario_label,
        "flood_id": flood_id,
        "color": color,
        "values": damage_values,
        "total_damage": float(event_damage_total_usd(dmg)),
        "n_links": int(len(damage_values)),
    })
    all_damage_values.append(damage_values)

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), sharey=True)

if len(scenario_damage) == 0:
    for ax in axes:
        ax.text(0.5, 0.5, "No scenario damage values available", ha="center", va="center")
        ax.set_axis_off()
else:
    all_values = pd.concat(all_damage_values, ignore_index=True) if len(all_damage_values) > 0 else pd.Series(dtype=float)
    bins = np.histogram_bin_edges(all_values, bins=40) if len(all_values) > 0 else np.linspace(0, 1, 40)

    for ax, item in zip(axes, scenario_damage):
        damage_values = item["values"]
        if len(damage_values) > 0:
            ax.hist(damage_values, bins=bins, color=item["color"], edgecolor="white", alpha=0.9)
        else:
            ax.text(0.5, 0.5, f"No values for {item['label']}", ha="center", va="center")
            ax.set_axis_off()
            continue

        ax.set_title(
            f"{item['label']} (id={item['flood_id']}) | Total Damage: {format_usd_millions(item['total_damage'])} | Links: {item['n_links']:,}",
            loc="left",
            fontsize=10,
        )
        ax.set_xlabel("Damage value (USD)")
        ax.grid(True, alpha=0.2)

    axes[0].set_ylabel("Count (links)")

fig.suptitle("Damage summary | direct damage distributions by scenario", fontsize=14)
fig.tight_layout()
save_figure(fig, step3_fig_dir, "step3_damage_summary_1_damage_distribution_panel")
plt.show()

C:\Users\akothaw\AppData\Local\Temp\ipykernel_37756\3930263031.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# Plot 2: total damage by road classification
fig, ax = plt.subplots(figsize=(11, 6))

classification_col = "road_classification_detail" if "road_classification_detail" in damage_step3.columns else "road_classification"
if classification_col in damage_step3.columns and not by_class.empty:
    ax.bar(by_class.index.astype(str), by_class.values, color="#54a24b")
    ax.set_title("Total damage by road classification")
    ax.set_xlabel("Road classification")
    ax.set_ylabel("Total damage (USD)")
    ax.tick_params(axis="x", rotation=45, labelsize=9)
    ax.grid(True, axis="y", alpha=0.2)
    # Clean up x-axis labels: wrap long names and improve readability
    labels = [label.get_text().replace("_", " ") for label in ax.get_xticklabels()]
    ax.set_xticklabels(labels, ha="right", fontsize=9)
else:
    ax.text(0.5, 0.5, "Road classification not available", ha="center", va="center")
    ax.set_axis_off()

fig.suptitle("Damage summary 2/4 | by road classification", fontsize=14)
fig.subplots_adjust(bottom=0.2, left=0.1, right=0.95, top=0.88)
save_figure(fig, step3_fig_dir, "step3_damage_summary_2_road_classification")
plt.show()

C:\Users\akothaw\AppData\Local\Temp\ipykernel_37756\592667605.py:14: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  ax.set_xticklabels(labels, ha="right", fontsize=9)
C:\Users\akothaw\AppData\Local\Temp\ipykernel_37756\592667605.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# Plot 3: spatial distribution of the most damaged links
top_damage = damage_per_link_value.sort_values("total_damage_value", ascending=False).head(50)
top50_total = float(top_damage["total_damage_value"].sum()) if len(top_damage) else 0.0
top50_max = float(top_damage["total_damage_value"].max()) if len(top_damage) else 0.0

fig, ax = plt.subplots(figsize=(10, 8))
if skip_map_layers:
    ax.text(
        0.5,
        0.5,
        f"Damage map omitted for large study area\n({len(road_links):,} links)",
        ha="center",
        va="center",
        fontsize=11,
    )
    ax.set_axis_off()
else:
    links_spatial = map_road_links.copy()
    links_spatial["e_id"] = links_spatial["e_id"].astype(str)
    top_damage_spatial = links_spatial.merge(
        top_damage[["e_id", "total_damage_value"]],
        on="e_id",
        how="inner",
    ).copy()
    if links_spatial.crs is not None and str(links_spatial.crs) != "EPSG:3857":
        map_df = links_spatial.to_crs("EPSG:3857")
        top_damage_spatial = (
            top_damage_spatial.to_crs("EPSG:3857") if len(top_damage_spatial) > 0 else top_damage_spatial
        )
    else:
        map_df = links_spatial
    ax.set_facecolor("#f5f7fa")
    map_df.plot(ax=ax, linewidth=1.4, color="#d7dce2", alpha=0.45, zorder=1)
    map_df.plot(ax=ax, linewidth=0.5, color="#aeb7c2", alpha=0.55, zorder=2)
    from matplotlib.lines import Line2D

    legend_elements = [Line2D([0], [0], color="#aeb7c2", lw=2, label="Network links")]
    if len(top_damage_spatial) > 0:
        vals = top_damage_spatial["total_damage_value"].astype(float).dropna().values
        if len(vals) > 1:
            edges = np.unique(np.quantile(vals, [0.0, 0.33, 0.66, 1.0]))
            if len(edges) < 4:
                vmin, vmax = float(np.min(vals)), float(np.max(vals))
                edges = (
                    np.linspace(vmin, vmax, 4)
                    if vmax > vmin
                    else np.array([vmin, vmin + 1e-6, vmin + 2e-6, vmin + 3e-6])
                )
        else:
            v = float(vals[0])
            edges = np.array([v, v + 1e-6, v + 2e-6, v + 3e-6])
        n_classes = len(edges) - 1
        width_levels = np.linspace(2.0, 6.0, n_classes)
        top_damage_spatial = top_damage_spatial.copy()
        top_damage_spatial["damage_class"] = pd.cut(
            top_damage_spatial["total_damage_value"],
            bins=edges,
            labels=False,
            include_lowest=True,
            duplicates="drop",
        )
        for i in range(n_classes):
            class_gdf = top_damage_spatial[top_damage_spatial["damage_class"] == i]
            if len(class_gdf) == 0:
                continue
            class_gdf.plot(ax=ax, linewidth=width_levels[i], color="#e63946", alpha=0.95, zorder=3)
            legend_elements.append(
                Line2D(
                    [0],
                    [0],
                    color="#e63946",
                    lw=float(width_levels[i]),
                    label=f"{edges[i] / 1000:,.1f}k to {edges[i + 1] / 1000:,.1f}k USD per link",
                )
            )
    else:
        ax.text(
            0.5,
            0.5,
            "No matching damaged links for spatial overlay",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
    ax.legend(
        handles=legend_elements,
        title="Map 3 legend (per-link SUM)",
        loc="upper left",
        frameon=True,
        framealpha=0.9,
        fontsize=8,
        title_fontsize=9,
    )
    ax.text(
        0.02,
        0.02,
        f"top50 total={top50_total / 1e6:.2f}M USD | max link={top50_max / 1000:.1f}k USD",
        transform=ax.transAxes,
        fontsize=8,
        color="#6b7280",
    )
    ax.set_title("Spatial distribution of most damaged links (top 50)")
    ax.set_axis_off()

fig.suptitle("Damage summary 3/4 | spatial distribution", fontsize=14)
fig.tight_layout()
save_figure(fig, step3_fig_dir, "step3_damage_summary_3_spatial_map")
plt.show()


C:\Users\akothaw\AppData\Local\Temp\ipykernel_37756\3600941348.py:108: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
# Plot 3a: spatial distribution of damage on ALL links for baseline / low / high scenarios
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import requests

scenario_specs = [("Baseline", 1), ("Low", 2), ("High", 3)]
scenario_maps = []
all_positive_damage = []

for scenario_label, flood_id in scenario_specs:
    links_path = results_root / "disruption_analysis" / VARIANT / str(DEPTH_KEY) / "links" / f"road_links_{flood_id}.gpq"
    dmg_path = results_root / "damage_analysis" / VARIANT / f"intersections_{flood_id}_with_damage_values.csv"
    if not links_path.exists() or not dmg_path.exists():
        print(f"Skipping {scenario_label}: missing link or damage file")
        continue

    links_gdf = subset_links_for_map(gpd.read_parquet(links_path).copy(), flow_col="current_flow")
    if "e_id" not in links_gdf.columns:
        print(f"Skipping {scenario_label}: no e_id column found")
        continue

    links_gdf["e_id"] = links_gdf["e_id"].astype(str)
    dmg = prepare_damage_for_viz(pd.read_csv(dmg_path))
    dmg["e_id"] = dmg["e_id"].astype(str)
    dmg["total_damage_value"] = pd.to_numeric(dmg["direct_damage_mean_usd"], errors="coerce").fillna(0.0)
    damage_per_link = dmg.groupby("e_id", as_index=False).agg(total_damage_value=("total_damage_value", "sum"))

    plot_gdf = links_gdf.merge(damage_per_link, on="e_id", how="left").copy()
    if plot_gdf.crs is not None and str(plot_gdf.crs) != "EPSG:3857":
        plot_gdf = plot_gdf.to_crs("EPSG:3857")
    scenario_maps.append({
        "label": scenario_label,
        "flood_id": flood_id,
        "gdf": plot_gdf,
    })
    vals = pd.to_numeric(plot_gdf["total_damage_value"], errors="coerce").dropna()
    if len(vals) > 0:
        all_positive_damage.append(vals[vals > 0])

fig, axes = plt.subplots(1, 3, figsize=(19, 5.8), constrained_layout=False)

if len(scenario_maps) == 0:
    for ax in axes:
        ax.text(0.5, 0.5, "No scenario maps available", ha="center", va="center")
        ax.set_axis_off()
else:
    all_values = pd.concat(all_positive_damage, ignore_index=True) if len(all_positive_damage) > 0 else pd.Series(dtype=float)
    vmax = float(all_values.quantile(0.99)) if len(all_values) > 1 else (float(all_values.max()) if len(all_values) else 1.0)
    vmax = max(vmax, 1.0)
    norm = Normalize(vmin=0, vmax=vmax)
    cmap = plt.get_cmap("magma")

    boundary = _load_va_boundary() if "_load_va_boundary" in globals() else None

    reference_geom = scenario_maps[0]["gdf"]
    if reference_geom.crs is None:
        reference_geom = reference_geom.set_crs("EPSG:3857")
    if boundary is not None and len(boundary) > 0:
        minx = min(reference_geom.total_bounds[0], boundary.total_bounds[0])
        miny = min(reference_geom.total_bounds[1], boundary.total_bounds[1])
        maxx = max(reference_geom.total_bounds[2], boundary.total_bounds[2])
        maxy = max(reference_geom.total_bounds[3], boundary.total_bounds[3])
    else:
        minx, miny, maxx, maxy = reference_geom.total_bounds

    pad_x = (maxx - minx) * 0.05 if maxx > minx else 1000
    pad_y = (maxy - miny) * 0.05 if maxy > miny else 1000
    extent = (minx - pad_x, maxx + pad_x, miny - pad_y, maxy + pad_y)

    for ax, item in zip(axes, scenario_maps):
        plot_gdf = item["gdf"]
        ax.set_facecolor("#f5f7fa")
        if skip_map_layers:
            ax.text(0.5, 0.5, "Scenario map omitted for large study area", ha="center", va="center")
            ax.set_axis_off()
            ax.set_title(f"{item['label']} (id={item['flood_id']})")
            continue
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
        ax.set_aspect("equal")

        plot_gdf.plot(ax=ax, linewidth=0.8, color="#d7dce2", alpha=0.35, zorder=1)
        plot_gdf.plot(
            ax=ax,
            column="total_damage_value",
            cmap=cmap,
            linewidth=1.1,
            alpha=0.95,
            vmin=0,
            vmax=vmax,
            legend=False,
            missing_kwds={"color": "#cbd5e1", "label": "No damage"},
            zorder=3,
        )

        if boundary is not None and len(boundary) > 0:
            boundary.boundary.plot(ax=ax, color="#111827", linewidth=1.8, zorder=4)

        ax.set_title(f"{item['label']} (id={item['flood_id']})")
        valid_vals = pd.to_numeric(plot_gdf["total_damage_value"], errors="coerce").dropna()
        ax.text(
            0.02,
            0.02,
            f"Total={float(valid_vals.sum()):,.2f} USD\nDamaged links={(valid_vals > 0).sum():,}",
            transform=ax.transAxes,
            fontsize=8,
            color="#334155",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.75, pad=2),
        )
        ax.set_axis_off()

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cax = fig.add_axes([0.32, 0.08, 0.36, 0.022])
    cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")
    cbar.set_label("Damage value (USD)")

fig.suptitle("Damage summary 3a/4 | spatial distribution (all links)", fontsize=14)
fig.subplots_adjust(left=0.02, right=0.98, top=0.88, bottom=0.18, wspace=0.03)
save_figure(fig, step3_fig_dir, "step3_damage_summary_3a_all_links_panel")
plt.show()


C:\Users\akothaw\AppData\Local\Temp\ipykernel_37756\3088043694.py:126: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
# Plot 4: damage fraction distribution (horizontal bars)
fig, ax = plt.subplots(figsize=(10, 5.5))

fraction_values = damage_per_link_fraction["total_damage_fraction"].dropna().astype(float)
if len(fraction_values) > 0:
    max_fraction = float(fraction_values.max()) if len(fraction_values) else 1.0
    max_fraction = max(max_fraction, 0.05)
    bins = np.linspace(0, max_fraction, 40)

    counts, edges = np.histogram(fraction_values, bins=bins)
    bin_centers = 0.5 * (edges[:-1] + edges[1:])

    baseline_color = SCENARIO_SPECS[0][2] if isinstance(SCENARIO_SPECS, (list, tuple)) and len(SCENARIO_SPECS) > 0 else "#72b7b2"
    ax.barh(bin_centers, counts, height=(edges[1] - edges[0]) * 0.9, color=baseline_color, edgecolor="white")

    ax.set_title("Damage fraction distribution")
    ax.set_xlabel("Count (links)")
    ax.set_ylabel("Damage fraction max (unitless)")
    ax.grid(True, axis="x", alpha=0.2)
else:
    ax.text(0.5, 0.5, "No damage fraction values available", ha="center", va="center")
    ax.set_axis_off()

fig.suptitle("Damage summary 4/4 | damage fractions", fontsize=14)
fig.tight_layout()
save_figure(fig, step3_fig_dir, "step3_damage_summary_4_damage_fractions")
plt.show()

C:\Users\akothaw\AppData\Local\Temp\ipykernel_37756\3838207019.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 2a. OD Origin-Attraction Hotspot Map

Visualize trip generation/attraction by centroid as a hotspot map to identify trip corridors and demand concentration.

In [14]:
# Step 2A: OD hotspot map in geographic coordinates with Virginia context + FAF5 underlay
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from shapely.geometry import Point
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

print("=" * 80)
print("STEP 2A: OD HOTSPOT MAP (GEOGRAPHIC) - VIRGINIA + FAF5 UNDERLAY")
print("=" * 80)

FAF5_GDB_PATH = Path(r"C:\Users\alimu\Desktop\Github\FAF5_Model_Highway_Network\Networks\Geodatabase Format\FAF5Network.gdb")


def _load_virginia_boundary() -> gpd.GeoDataFrame | None:
    """Load Virginia boundary from local files if possible, with Census fallback."""
    candidate_files = []

    # Search likely local locations under input_root
    for sub in ["study_area", "census_datasets"]:
        p = Path(input_root) / sub
        if p.exists():
            for ext in ("*.shp", "*.gpkg", "*.geojson"):
                candidate_files.extend(list(p.rglob(ext)))

    # Prioritize filenames likely to contain state polygons
    ranked = sorted(
        candidate_files,
        key=lambda x: (
            0 if "state" in x.name.lower() or "virginia" in x.name.lower() or "va" in x.name.lower() else 1,
            len(x.name),
        ),
    )

    for f in ranked:
        try:
            gdf = gpd.read_file(f)
            cols = {c.lower(): c for c in gdf.columns}

            # Try common state columns
            if "stusps" in cols:
                va = gdf[gdf[cols["stusps"]].astype(str).str.upper() == "VA"]
                if len(va) > 0:
                    print(f"[OK] Virginia boundary loaded from local file: {f}")
                    return va

            if "state" in cols:
                s = gdf[cols["state"]].astype(str).str.upper()
                va = gdf[s.isin(["VA", "VIRGINIA"]) ]
                if len(va) > 0:
                    print(f"[OK] Virginia boundary loaded from local file: {f}")
                    return va

            if "name" in cols:
                va = gdf[gdf[cols["name"]].astype(str).str.upper() == "VIRGINIA"]
                if len(va) > 0:
                    print(f"[OK] Virginia boundary loaded from local file: {f}")
                    return va
        except Exception:
            continue

    # Census fallback (URL zip shapefile)
    census_url = "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_state_20m.zip"
    try:
        us_states = gpd.read_file(census_url)
        if "STUSPS" in us_states.columns:
            va = us_states[us_states["STUSPS"] == "VA"].copy()
            if len(va) > 0:
                print("[OK] Virginia boundary loaded from Census URL")
                return va
    except Exception:
        pass

    print("[WARNING] Could not load Virginia boundary; using data extent only")
    return None


# 1) Load OD matrix
if skip_map_layers:
    print("Skipping Step 2A OD hotspot map for large study area.")
else:
    try:
        odpfc, od_path = load_odpfc(results_variant_root)
    except FileNotFoundError:
        odpfc, od_path = assignment_od, assignment_od_source
    print(f"[OK] OD matrix rows: {len(odpfc):,}")
    print(f"[OK] Unique OD centroids: {odpfc['origin_node'].nunique()}")


    # 2) Load local modeled road network for centroid node coordinates (node IDs align with OD)
    road_links = gpd.read_parquet(paths["disruption_links"])
    print(f"[OK] Modeled road links loaded: {len(road_links):,}")

    node_coords = {}
    has_node_cols = {"from_id", "to_id"}.issubset(road_links.columns)
    if not has_node_cols:
        raise ValueError("road_links does not contain from_id/to_id needed to map OD nodes")

    for _, row in road_links.iterrows():
        geom = row["geometry"]
        from_id = str(row["from_id"])
        to_id = str(row["to_id"])

        if geom.geom_type == "LineString":
            coords = list(geom.coords)
            if coords:
                node_coords[from_id] = coords[0]
                node_coords[to_id] = coords[-1]
        elif geom.geom_type == "MultiLineString":
            # Use first segment start and last segment end
            lines = list(geom.geoms)
            if len(lines) > 0:
                c0 = list(lines[0].coords)
                c1 = list(lines[-1].coords)
                if c0:
                    node_coords[from_id] = c0[0]
                if c1:
                    node_coords[to_id] = c1[-1]

    print(f"[OK] Extracted node coordinates: {len(node_coords):,}")


    # 3) Build centroid points for OD nodes
    od_nodes = set(odpfc["origin_node"].astype(str).unique())
    centroid_points = []
    for nid in od_nodes:
        if nid in node_coords:
            x, y = node_coords[nid]
            centroid_points.append({"node_id": nid, "geometry": Point(x, y)})

    centroids_gdf = gpd.GeoDataFrame(centroid_points, crs=road_links.crs)
    print(f"[OK] Centroid points with geometry: {len(centroids_gdf)} / {len(od_nodes)}")


    # 4) Aggregate generation + attraction
    gen_flows = odpfc.groupby("origin_node", as_index=False)["flow"].sum().rename(
        columns={"origin_node": "node_id", "flow": "generated"}
    )
    attr_flows = odpfc.groupby("destination_node", as_index=False)["flow"].sum().rename(
        columns={"destination_node": "node_id", "flow": "attracted"}
    )

    flow_agg = gen_flows.merge(attr_flows, on="node_id", how="outer").fillna(0)
    flow_agg["node_id"] = flow_agg["node_id"].astype(str)
    flow_agg["total_flow"] = flow_agg["generated"] + flow_agg["attracted"]

    hotspot_gdf = centroids_gdf.merge(flow_agg, on="node_id", how="left").fillna(0)
    hotspot_gdf = hotspot_gdf[hotspot_gdf["total_flow"] > 0].copy()
    print(f"[OK] Hotspot centroids plotted: {len(hotspot_gdf)}")


    # 5) Load Virginia boundary + FAF5_Links underlay and convert all to geographic coordinates
    va_boundary = _load_virginia_boundary()

    faf5_links = None
    if FAF5_GDB_PATH.exists():
        try:
            faf5_links = gpd.read_file(FAF5_GDB_PATH, layer="FAF5_Links")
            print(f"[OK] FAF5_Links loaded from GDB: {len(faf5_links):,}")
        except Exception as e:
            print(f"[WARNING] Could not load FAF5_Links from GDB: {e}")
    else:
        print(f"[WARNING] FAF5 GDB not found at: {FAF5_GDB_PATH}")

    # Convert to EPSG:4326 (geographic)
    hotspot_geo = hotspot_gdf.to_crs("EPSG:4326") if hotspot_gdf.crs and str(hotspot_gdf.crs) != "EPSG:4326" else hotspot_gdf.copy()
    va_geo = None if va_boundary is None else (va_boundary.to_crs("EPSG:4326") if va_boundary.crs and str(va_boundary.crs) != "EPSG:4326" else va_boundary.copy())
    faf5_geo = None
    if faf5_links is not None:
        faf5_geo = faf5_links.to_crs("EPSG:4326") if faf5_links.crs and str(faf5_links.crs) != "EPSG:4326" else faf5_links.copy()

    # Clip FAF5 links to Virginia polygon for cleaner plot
    if faf5_geo is not None and va_geo is not None and len(va_geo) > 0:
        try:
            va_union = va_geo.geometry.union_all() if hasattr(va_geo.geometry, "union_all") else va_geo.unary_union
            faf5_geo = faf5_geo[faf5_geo.intersects(va_union)].copy()
            print(f"[OK] FAF5_Links after VA clip: {len(faf5_geo):,}")
        except Exception:
            pass


    # 6) Plot maps with Virginia context + FAF5 underlay + centroid labels
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 11), constrained_layout=True)

    for ax in (ax1, ax2):
        ax.set_facecolor("#f8fafc")

        # Virginia boundary context
        if va_geo is not None and len(va_geo) > 0:
            va_geo.boundary.plot(ax=ax, color="#111827", linewidth=1.2, zorder=1)
            va_geo.plot(ax=ax, color="#f1f5f9", alpha=0.35, zorder=0)

        # FAF5 road network underlay (requested soft alpha=0.7)
        if faf5_geo is not None and len(faf5_geo) > 0:
            faf5_geo.plot(ax=ax, color="#94a3b8", linewidth=0.35, alpha=0.7, zorder=2)

    # Generation map
    gen_max = max(float(hotspot_geo["generated"].max()), 1.0)
    sizes_gen = (hotspot_geo["generated"] / gen_max) * 360 + 24
    sc1 = ax1.scatter(
        hotspot_geo.geometry.x,
        hotspot_geo.geometry.y,
        s=sizes_gen,
        c=hotspot_geo["generated"],
        cmap="Reds",
        norm=Normalize(vmin=0, vmax=gen_max),
        alpha=0.78,
        edgecolors="#991b1b",
        linewidth=1.0,
        zorder=4,
    )
    ax1.set_title("Trip Generation Hotspots (Virginia, geographic)", fontsize=14, fontweight="bold")
    ax1.set_xlabel("Longitude")
    ax1.set_ylabel("Latitude")
    ax1.grid(True, alpha=0.2)
    cbar1 = plt.colorbar(sc1, ax=ax1, shrink=0.82)
    cbar1.set_label("Trip generation (vehicles)")

    # Attraction map
    attr_max = max(float(hotspot_geo["attracted"].max()), 1.0)
    sizes_attr = (hotspot_geo["attracted"] / attr_max) * 360 + 24
    sc2 = ax2.scatter(
        hotspot_geo.geometry.x,
        hotspot_geo.geometry.y,
        s=sizes_attr,
        c=hotspot_geo["attracted"],
        cmap="Blues",
        norm=Normalize(vmin=0, vmax=attr_max),
        alpha=0.78,
        edgecolors="#1e3a8a",
        linewidth=1.0,
        zorder=4,
    )
    ax2.set_title("Trip Attraction Hotspots (Virginia, geographic)", fontsize=14, fontweight="bold")
    ax2.set_xlabel("Longitude")
    ax2.set_ylabel("Latitude")
    ax2.grid(True, alpha=0.2)
    cbar2 = plt.colorbar(sc2, ax=ax2, shrink=0.82)
    cbar2.set_label("Trip attraction (vehicles)")

    # Add centroid IDs on both panels (requested)
    for _, row in hotspot_geo.iterrows():
        x = row.geometry.x
        y = row.geometry.y
        label = str(row["node_id"])

        ax1.annotate(
            label,
            (x, y),
            xytext=(2, 2),
            textcoords="offset points",
            fontsize=6,
            color="#7f1d1d",
            alpha=0.9,
            zorder=5,
        )
        ax2.annotate(
            label,
            (x, y),
            xytext=(2, 2),
            textcoords="offset points",
            fontsize=6,
            color="#1e3a8a",
            alpha=0.9,
            zorder=5,
        )

    # Set extent to VA boundary when available, else to hotspot bounds
    if va_geo is not None and len(va_geo) > 0:
        minx, miny, maxx, maxy = va_geo.total_bounds
    else:
        minx, miny, maxx, maxy = hotspot_geo.total_bounds

    pad_x = (maxx - minx) * 0.05 if maxx > minx else 0.2
    pad_y = (maxy - miny) * 0.05 if maxy > miny else 0.2
    for ax in (ax1, ax2):
        ax.set_xlim(minx - pad_x, maxx + pad_x)
        ax.set_ylim(miny - pad_y, maxy + pad_y)

    # Save figure
    step2a_fig_dir = fig_root / "step2_disruption"
    step2a_fig_dir.mkdir(parents=True, exist_ok=True)
    fig_path = step2a_fig_dir / "OD_hotspot_map_geographic_va_with_faf5_underlay.png"
    fig.savefig(fig_path, dpi=170, bbox_inches="tight")
    print(f"[OK] Saved geographic hotspot map: {fig_path}")

    plt.show()

    # Quick top-centroid table
    top_gen = hotspot_gdf.nlargest(15, "generated")[["node_id", "generated", "attracted", "total_flow"]].copy()
    print("\nTop 15 generation centroids:")
    print(top_gen.to_string(index=False, float_format=lambda v: f"{v:,.0f}"))


STEP 2A: OD HOTSPOT MAP (GEOGRAPHIC) - VIRGINIA + FAF5 UNDERLAY
Skipping Step 2A OD hotspot map for large study area.


In [15]:
# Step 2A check: verify OD trips vs Script 1 network flow totals
import pandas as pd

# Load OD matrix and Script 1 edge flow outputs directly from canonical paths
try:
    od_check, od_check_source = load_odpfc(results_variant_root)
except FileNotFoundError:
    od_check, od_check_source = assignment_od, assignment_od_source
edge_check, edge_check_source = load_edge_flows(results_variant_root)

# Identify flow columns
od_flow_col = "flow" if "flow" in od_check.columns else ("Car21" if "Car21" in od_check.columns else None)
net_flow_col = "acc_flow" if "acc_flow" in edge_check.columns else ("flow" if "flow" in edge_check.columns else None)

if od_flow_col is None:
    raise ValueError("No OD flow column found in odpfc.pq (expected 'flow' or 'Car21').")
if net_flow_col is None:
    raise ValueError("No network flow column found in edge_flows.gpq (expected 'acc_flow' or 'flow').")

# Totals
od_total_trips = float(pd.to_numeric(od_check[od_flow_col], errors="coerce").fillna(0.0).sum())
network_total_flow = float(pd.to_numeric(edge_check[net_flow_col], errors="coerce").fillna(0.0).sum())

# Useful diagnostics
od_pairs = int(len(od_check))
unique_origins = int(od_check["origin_node"].nunique()) if "origin_node" in od_check.columns else None
unique_destinations = int(od_check["destination_node"].nunique()) if "destination_node" in od_check.columns else None
ratio = network_total_flow / od_total_trips if od_total_trips > 0 else float("nan")

print("=" * 80)
print("VERIFICATION: TOTAL TRIPS (OD) VS TOTAL NETWORK FLOW (SCRIPT 1)")
print("=" * 80)
print(f"OD matrix file             : {od_check_source}")
print(f"Edge flow file             : {edge_check_source}")
print(f"OD flow column used        : {od_flow_col}")
print(f"Network flow column used   : {net_flow_col}")
print("-")
print(f"OD pairs                   : {od_pairs:,}")
if unique_origins is not None:
    print(f"Unique origins             : {unique_origins:,}")
if unique_destinations is not None:
    print(f"Unique destinations        : {unique_destinations:,}")
print("-")
print(f"Total trips from OD matrix : {od_total_trips:,.3f}")
print(f"Total flow on network      : {network_total_flow:,.3f}")
print(f"Network/OD ratio           : {ratio:,.3f}x")
print("=")

VERIFICATION: TOTAL TRIPS (OD) VS TOTAL NETWORK FLOW (SCRIPT 1)
OD matrix file             : C:\Users\akothaw\Desktop\data\results\base_scenario\revision\odpfc.pq
Edge flow file             : C:\Users\akothaw\Desktop\data\results\base_scenario\revision\edge_flows_pass_a.gpq
OD flow column used        : flow
Network flow column used   : acc_flow
-
OD pairs                   : 19,351
Unique origins             : 3,123
Unique destinations        : 3,122
-
Total trips from OD matrix : 3,319.617
Total flow on network      : 3,234,089.572
Network/OD ratio           : 974.236x
=


In [16]:
od_check.head()
od_check.flow.sum()

np.float64(3319.6170142758883)

In [17]:
# Plot 4a: damage fraction distributions for baseline / low / high scenarios (horizontal bars)
scenario_specs = SCENARIO_SPECS
scenario_fractions = []
all_fraction_values = []

for scenario_label, flood_id, color in scenario_specs:
    dmg_path = results_root / "damage_analysis" / VARIANT / f"intersections_{flood_id}_with_damage_values.csv"
    if not dmg_path.exists():
        print(f"Skipping {scenario_label}: missing {dmg_path}")
        continue

    dmg = pd.read_csv(dmg_path)
    damage_fraction_cols = [c for c in dmg.columns if c.endswith("_damage_fraction")]
    if not damage_fraction_cols:
        print(f"Skipping {scenario_label}: no damage fraction columns in {dmg_path.name}")
        continue

    fraction_values = dmg[damage_fraction_cols].apply(pd.to_numeric, errors="coerce").max(axis=1).dropna().astype(float)
    scenario_fractions.append({
        "label": scenario_label,
        "flood_id": flood_id,
        "color": color,
        "values": fraction_values,
        "mean_fraction": float(fraction_values.mean()) if len(fraction_values) else 0.0,
        "max_fraction": float(fraction_values.max()) if len(fraction_values) else 0.0,
        "n_links": int(len(fraction_values)),
    })
    all_fraction_values.append(fraction_values)

fig, axes = plt.subplots(3, 1, figsize=(6, 12), sharex=False, sharey=False)

if len(scenario_fractions) == 0:
    for ax in axes:
        ax.text(0.5, 0.5, "No scenario fraction values available", ha="center", va="center")
        ax.set_axis_off()
else:
    all_values = pd.concat(all_fraction_values, ignore_index=True) if len(all_fraction_values) > 0 else pd.Series(dtype=float)
    max_fraction = float(all_values.max()) if len(all_values) else 1.0
    max_fraction = max(max_fraction, 0.05)
    bins = np.linspace(0, max_fraction, 90)

    for ax, item in zip(axes, scenario_fractions):
        fraction_values = item["values"]
        if len(fraction_values) > 0:
            counts, edges = np.histogram(fraction_values, bins=bins)
            bin_centers = 0.5 * (edges[:-1] + edges[1:])
            ax.barh(bin_centers, counts, height=(edges[1] - edges[0]) * 0.9, color=item["color"], edgecolor="white")
            ax.text(
                0.98,
                0.98,
                f"Mean: {item['mean_fraction']:.3f} | Max: {item['max_fraction']:.3f} | Links: {item['n_links']:,}",
                transform=ax.transAxes,
                ha="right",
                va="top",
                fontsize=8,
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.75, pad=2),
            )
        else:
            ax.text(0.5, 0.5, f"No values for {item['label']}", ha="center", va="center")
            ax.set_axis_off()
            continue

        ax.set_ylabel("Damage fraction max (unitless)")
        ax.grid(True, axis="x", alpha=0.2)

# show count on x-axis for all subplots; add xlabel to bottom
axes[-1].set_xlabel("Count (links)")

fig.suptitle("Damage summary 4a/4 | damage fractions by scenario", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.97])
save_figure(fig, step3_fig_dir, "step3_damage_summary_4a_damage_fractions_panel")
plt.show()

### Step 3A. Per-link aggregation check (sum vs max)
This diagnostic compares two ways to collapse multiple damage rows per link (`e_id`):
- **Sum**: treats each row as an additive damage contribution
- **Max**: treats rows as alternative estimates and keeps the worst-case per link

## Step 4. Rerouting and recovery visuals

In [18]:
def _available_rerouting_event_ids(variant_name: str, depth_key: int) -> list[int]:
    rerouting_root = results_root / "rerouting_analysis" / variant_name / str(depth_key)
    if not rerouting_root.exists():
        return []
    event_ids = []
    for child in rerouting_root.iterdir():
        if child.is_dir() and child.name.isdigit():
            event_ids.append(int(child.name))
    return sorted(set(event_ids))


def _load_rerouting_costs_for_event(event_id: int) -> pd.DataFrame:
    rerouting_path = results_root / "rerouting_analysis" / VARIANT / str(DEPTH_KEY) / str(event_id) / "cost_matrix_by_scenario.csv"
    if not rerouting_path.exists():
        return pd.DataFrame()
    return pd.read_csv(rerouting_path).sort_values("scenario")


available_rerouting_event_ids = _available_rerouting_event_ids(VARIANT, DEPTH_KEY)
rerouting_source_event_id = None
rerouting_costs_all = pd.DataFrame()

if not rerouting_costs.empty and "scenario" in rerouting_costs.columns:
    rerouting_costs_all = rerouting_costs.copy().sort_values("scenario")
    rerouting_source_event_id = FLOOD_KEY if (paths["rerouting_costs"].exists()) else None
else:
    for event_id in available_rerouting_event_ids:
        candidate = _load_rerouting_costs_for_event(event_id)
        if not candidate.empty and "scenario" in candidate.columns:
            rerouting_costs_all = candidate.copy().sort_values("scenario")
            rerouting_source_event_id = event_id
            break

if rerouting_costs_all.empty:
    print("Rerouting outputs not found for this results set; skipping Step 4 summary.")
else:
    # Keep a copy of the full rerouting table for downstream cells, but make this cell baseline-only.
    baseline_scenario_value = int(pd.to_numeric(rerouting_costs_all["scenario"], errors="coerce").dropna().min())
    rerouting_costs = rerouting_costs_all.loc[pd.to_numeric(rerouting_costs_all["scenario"], errors="coerce") == baseline_scenario_value].copy()
    if rerouting_costs.empty:
        print("No baseline row found in rerouting outputs; skipping Step 4 summary.")
    else:
        step4 = rerouting_costs.copy().sort_values("scenario")
        scenario_days = {idx: day for idx, day in enumerate(RECOVERY_DAYS)}
        step4["event_day"] = step4["scenario"].map(scenario_days)

        trip_rows = []
        trip_flows = []
        for s in step4["scenario"].astype(int):
            pq_path = paths["rerouting_dir"] / f"trip_isolations_{s}.pq"
            csv_path = paths["rerouting_dir"] / f"trip_isolations_{s}.csv"
            p = pq_path if pq_path.exists() else csv_path
            if p.exists():
                df = pd.read_parquet(p) if p.suffix == ".pq" else pd.read_csv(p)
                fc = "flow" if "flow" in df.columns else ("Car21" if "Car21" in df.columns else None)
                trip_rows.append(len(df))
                trip_flows.append(float(df[fc].sum()) if fc else float(len(df)))
            else:
                trip_rows.append(np.nan)
                trip_flows.append(np.nan)

        step4["isolated_trip_rows"] = trip_rows
        step4["isolated_trip_flow"] = trip_flows

        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        fig._nird_plot_scenario = RUN_CONTEXT["baseline_scenario_label"]
        axes = axes.flatten()
        x = step4["scenario"].astype(int)

        axes[0].plot(x, step4["rerouting_cost"], marker="o", color="#4c78a8")
        axes[0].set_title("Baseline rerouting cost")
        axes[0].set_xlabel("Scenario")
        axes[0].set_ylabel("Rerouting cost")
        axes[0].grid(True, alpha=0.2)

        axes[1].plot(x, step4["direct_damage_total"], marker="o", color="#f58518")
        axes[1].set_title("Baseline direct damage")
        axes[1].set_xlabel("Scenario")
        axes[1].set_ylabel("Direct damage (USD)")
        axes[1].grid(True, alpha=0.2)

        axes[2].plot(x, step4["combined_total_cost"], marker="o", color="#54a24b")
        axes[2].set_title("Baseline combined total cost")
        axes[2].set_xlabel("Scenario")
        axes[2].set_ylabel("Combined total cost")
        axes[2].grid(True, alpha=0.2)

        axes[3].plot(x, step4["isolated_trip_flow"], marker="o", color="#b279a2")
        axes[3].set_title("Baseline isolated trip flow")
        axes[3].set_xlabel("Scenario")
        axes[3].set_ylabel("Isolated flow")
        axes[3].grid(True, alpha=0.2)

        labels = [f"S{s}\nD{d}" for s, d in zip(step4["scenario"].astype(int), step4["event_day"].astype(int))]
        for ax in axes:
            ax.set_xticks(x)
            ax.set_xticklabels(labels, fontsize=8)

        fig.suptitle(f"Recovery summary | baseline only (source event={rerouting_source_event_id})", fontsize=14)
        fig.tight_layout()
        save_figure(fig, step4_fig_dir, "step4_recovery_summary_baseline_only")
        plt.show()

### Step 4A. Cost assumptions and trip accounting context
This diagnostic panel helps interpret rerouting numbers:
- `rerouting_cost = rer_time + rer_operate + rer_toll` (all are deltas vs baseline)
- negative rerouting cost means net travel-cost savings relative to baseline
- isolated trip flow is taken from `trip_isolations_{scenario}` outputs
- isolated share is computed as `isolated_trip_flow / baseline_total_flow`

In [19]:
if rerouting_costs.empty or "scenario" not in rerouting_costs.columns:
    print("Rerouting outputs not found for this results set; skipping Step 4A interpretation panel.")
else:
    # Build an interpretation panel for rerouting economics and trip accounting
    step4_diag = rerouting_costs.copy().sort_values("scenario").reset_index(drop=True)
    step4_diag["scenario"] = step4_diag["scenario"].astype(int)

    # Baseline flow denominator for isolation share
    flow_col_diag = "flow" if "flow" in odpfc.columns else ("Car21" if "Car21" in odpfc.columns else None)
    baseline_total_flow = float(odpfc[flow_col_diag].sum()) if flow_col_diag else np.nan

    # Bring isolated trip rows/flow into this diagnostic dataframe
    iso_rows = []
    iso_flows = []
    for s in step4_diag["scenario"]:
        pq_path = paths["rerouting_dir"] / f"trip_isolations_{s}.pq"
        csv_path = paths["rerouting_dir"] / f"trip_isolations_{s}.csv"
        p = pq_path if pq_path.exists() else csv_path
        if p.exists():
            iso_df = pd.read_parquet(p) if p.suffix == ".pq" else pd.read_csv(p)
            fc = "flow" if "flow" in iso_df.columns else ("Car21" if "Car21" in iso_df.columns else None)
            iso_rows.append(int(len(iso_df)))
            iso_flows.append(float(iso_df[fc].sum()) if fc else float(len(iso_df)))
        else:
            iso_rows.append(np.nan)
            iso_flows.append(np.nan)

    step4_diag["isolated_trip_rows"] = iso_rows
    step4_diag["isolated_trip_flow"] = iso_flows
    step4_diag["isolated_share_pct"] = (step4_diag["isolated_trip_flow"] / baseline_total_flow * 100.0) if baseline_total_flow > 0 else np.nan

    # Split rerouting into penalty (positive) vs savings (negative)
    step4_diag["rerouting_penalty"] = step4_diag["rerouting_cost"].clip(lower=0)
    step4_diag["rerouting_saving"] = step4_diag["rerouting_cost"].clip(upper=0)

    # Consistency check: combined should equal direct + rerouting
    step4_diag["combined_check"] = step4_diag["direct_damage_total"] + step4_diag["rerouting_cost"]
    step4_diag["combined_residual"] = step4_diag["combined_total_cost"] - step4_diag["combined_check"]

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()
    x = step4_diag["scenario"].to_numpy()

    # A) Rerouting components (delta vs baseline)
    axes[0].bar(x, step4_diag["rer_time"], label="Time delta", color="#4c78a8")
    axes[0].bar(x, step4_diag["rer_operate"], bottom=step4_diag["rer_time"], label="Operating delta", color="#f58518")
    axes[0].bar(x, step4_diag["rer_toll"], bottom=step4_diag["rer_time"] + step4_diag["rer_operate"], label="Toll delta", color="#54a24b")
    axes[0].axhline(0, color="black", linewidth=0.8)
    axes[0].set_title("Rerouting cost components (delta vs baseline)")
    axes[0].set_xlabel("Scenario")
    axes[0].set_ylabel("Cost delta")
    axes[0].legend(frameon=False, fontsize=8)
    axes[0].grid(True, axis="y", alpha=0.2)

    # B) Isolated trip accounting
    axes[1].bar(x, step4_diag["isolated_trip_flow"], color="#b279a2", alpha=0.8, label="Isolated trip flow")
    ax1b = axes[1].twinx()
    ax1b.plot(x, step4_diag["isolated_share_pct"], color="#e45756", marker="o", label="Isolated share (%)")
    axes[1].set_title("How many trips are isolated?")
    axes[1].set_xlabel("Scenario")
    axes[1].set_ylabel("Isolated trip flow")
    ax1b.set_ylabel("Isolated share of baseline flow (%)")
    axes[1].grid(True, axis="y", alpha=0.2)

    # combined legend for twin axis
    h1, l1 = axes[1].get_legend_handles_labels()
    h2, l2 = ax1b.get_legend_handles_labels()
    axes[1].legend(h1 + h2, l1 + l2, frameon=False, fontsize=8, loc="upper right")

    # C) Rerouting penalty vs savings
    axes[2].bar(x, step4_diag["rerouting_penalty"], color="#72b7b2", label="Penalty (+)")
    axes[2].bar(x, step4_diag["rerouting_saving"], color="#e15759", label="Saving (-)")
    axes[2].axhline(0, color="black", linewidth=0.8)
    axes[2].set_title("Rerouting net effect (penalty vs saving)")
    axes[2].set_xlabel("Scenario")
    axes[2].set_ylabel("Cost delta")
    axes[2].legend(frameon=False, fontsize=8)
    axes[2].grid(True, axis="y", alpha=0.2)

    # D) Direct vs rerouting vs combined
    axes[3].plot(x, step4_diag["direct_damage_total"], marker="o", color="#f58518", label="Direct damage (USD)")
    axes[3].plot(x, step4_diag["rerouting_cost"], marker="o", color="#4c78a8", label="Rerouting delta")
    axes[3].plot(x, step4_diag["combined_total_cost"], marker="o", color="#54a24b", label="Combined total")
    axes[3].set_title("Cost stack context")
    axes[3].set_xlabel("Scenario")
    axes[3].set_ylabel("Cost")
    axes[3].legend(frameon=False, fontsize=8)
    axes[3].grid(True, alpha=0.2)

    labels = [f"S{s}\nD{d}" for s, d in zip(step4_diag["scenario"].astype(int), step4_diag["scenario"].map({idx: day for idx, day in enumerate(RECOVERY_DAYS)}).astype(int))]
    for ax in [axes[0], axes[1], axes[2], axes[3]]:
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=8)

    fig.suptitle(
        f"Step 4A interpretation | baseline flow={baseline_total_flow:,.0f} | max |combined residual|={step4_diag['combined_residual'].abs().max():.4g}",
        fontsize=13,
    )
    fig.tight_layout()
    save_figure(fig, step4_fig_dir, "step4_interpretation_panel")

    # Compact numeric table for quick review
    display_cols = [
        "scenario",
        "rer_time",
        "rer_operate",
        "rer_toll",
        "rerouting_cost",
        "isolated_trip_rows",
        "isolated_trip_flow",
        "isolated_share_pct",
        "direct_damage_total",
        "combined_total_cost",
    ]
    print(step4_diag[display_cols].to_string(index=False, float_format=lambda v: f"{v:,.3f}"))

    plt.show()

 scenario  rer_time  rer_operate  rer_toll  rerouting_cost  isolated_trip_rows  isolated_trip_flow  isolated_share_pct  direct_damage_total  combined_total_cost
        1     0.000        0.000     0.000           0.000                   0               0.000               0.000                2.735                2.735


In [20]:
# Step 4C. Rerouting summary for Baseline / Low / High event folders
# This mirrors the other scenario small-multiple plots above and compares the available rerouting outputs side by side.

from pathlib import Path


def _available_rerouting_event_ids(variant_name: str, depth_key: int) -> list[int]:
    rerouting_root = results_root / "rerouting_analysis" / variant_name / str(depth_key)
    if not rerouting_root.exists():
        return []
    event_ids = []
    for child in rerouting_root.iterdir():
        if child.is_dir() and child.name.isdigit():
            event_ids.append(int(child.name))
    return sorted(set(event_ids))


def _load_rerouting_costs_for_event(event_id: int) -> pd.DataFrame:
    rerouting_path = results_root / "rerouting_analysis" / VARIANT / str(DEPTH_KEY) / str(event_id) / "cost_matrix_by_scenario.csv"
    if not rerouting_path.exists():
        return pd.DataFrame()
    return pd.read_csv(rerouting_path).sort_values("scenario")


def _load_trip_isolations_for_event(event_id: int, scenario_id: int) -> pd.DataFrame:
    pq_path = results_root / "rerouting_analysis" / VARIANT / str(DEPTH_KEY) / str(event_id) / f"trip_isolations_{scenario_id}.pq"
    csv_path = results_root / "rerouting_analysis" / VARIANT / str(DEPTH_KEY) / str(event_id) / f"trip_isolations_{scenario_id}.csv"
    path = pq_path if pq_path.exists() else csv_path
    if not path.exists():
        return pd.DataFrame()
    return pd.read_parquet(path) if path.suffix == ".pq" else pd.read_csv(path)


available_event_ids = _available_rerouting_event_ids(VARIANT, DEPTH_KEY)

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), sharey=False)

for ax, (scenario_label, event_id, color) in zip(axes, SCENARIO_SPECS):
    cost_df = _load_rerouting_costs_for_event(event_id)
    if cost_df.empty:
        ax.text(0.5, 0.5, f"Missing rerouting outputs\n{scenario_label} (event={event_id})", ha="center", va="center")
        ax.set_axis_off()
        continue

    cost_df = cost_df.copy()
    cost_df["scenario"] = pd.to_numeric(cost_df["scenario"], errors="coerce")
    cost_df = cost_df.dropna(subset=["scenario"]).sort_values("scenario")
    cost_df["scenario"] = cost_df["scenario"].astype(int)

    ax.plot(cost_df["scenario"], cost_df["combined_total_cost"], marker="o", color=color, label="Combined total")
    ax.plot(cost_df["scenario"], cost_df["rerouting_cost"], marker="o", color="#4c78a8", linestyle="--", label="Rerouting cost")
    ax.plot(cost_df["scenario"], cost_df["direct_damage_total"], marker="o", color="#f58518", linestyle=":", label="Direct damage (USD)")

    first_scenario = int(cost_df["scenario"].min()) if len(cost_df) else 0
    trip_df = _load_trip_isolations_for_event(event_id, first_scenario)
    ax2 = None
    if not trip_df.empty:
        flow_col_local = "flow" if "flow" in trip_df.columns else ("Car21" if "Car21" in trip_df.columns else None)
        isolated_flow = float(pd.to_numeric(trip_df[flow_col_local], errors="coerce").fillna(0.0).sum()) if flow_col_local else float(len(trip_df))
        ax2 = ax.twinx()
        ax2.bar([first_scenario], [isolated_flow], color="#b279a2", alpha=0.18, width=0.6, label="Isolated flow")
        ax2.set_ylabel("Isolated flow")
        ax2.tick_params(axis="y")

    ax.set_title(f"{scenario_label} (event={event_id})")
    ax.set_xlabel("Recovery scenario")
    ax.set_ylabel("USD")
    ax.grid(True, axis="y", alpha=0.2)
    handles, labels = ax.get_legend_handles_labels()
    if ax2 is not None:
        handles2, labels2 = ax2.get_legend_handles_labels()
        handles += handles2
        labels += labels2
    ax.legend(handles, labels, frameon=False, fontsize=8, loc="upper left")

fig.suptitle("Recovery summary across Baseline / Low / High event folders", fontsize=14)
fig.tight_layout()
save_figure(fig, step4_fig_dir, "step4_recovery_summary_all_three_events")
plt.show()

print("Available rerouting event folders:", available_event_ids)


Available rerouting event folders: [1, 2, 3]


In [21]:
# Step 4B. Base-scenario link V/C (volume/capacity) visualization
# Uses base scenario edge flows from script 1 only.
# This cell is self-contained so it can run even after a kernel restart.

from pathlib import Path
import os

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ---- bootstrap context if prior cells were not executed ----
if "VARIANT" not in globals():
    VARIANT = os.getenv("NIRD_RESULTS_VARIANT", "od_inflation_100x")
if "DEPTH_KEY" not in globals():
    DEPTH_KEY = 30
if "FLOOD_KEY" not in globals():
    FLOOD_KEY = 1

if "root" not in globals():
    root = Path.cwd()
    while root != root.parent and not (root / "pyproject.toml").exists():
        root = root.parent

if "results_root" not in globals():
    results_root = root / "sandbox" / "results"
if "fig_root" not in globals():
    fig_root = results_root / "figures" / VARIANT
if "step4_fig_dir" not in globals():
    step4_fig_dir = fig_root / "step4_rerouting"
step4_fig_dir.mkdir(parents=True, exist_ok=True)

if "save_figure" not in globals():
    def save_figure(fig: plt.Figure, out_dir: Path, stem: str) -> None:
        out_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(out_dir / f"{stem}.png", dpi=300, bbox_inches="tight", facecolor="white")
        fig.savefig(out_dir / f"{stem}.pdf", bbox_inches="tight", facecolor="white")

# ---- base scenario edge flows from script 1 ----
base_edge_path = results_root / "base_scenario" / VARIANT / "edge_flows.gpq"
if not base_edge_path.exists():
    raise FileNotFoundError(f"Missing base scenario edge flow file: {base_edge_path}")

final_edge_path = base_edge_path
final_edges = gpd.read_parquet(final_edge_path).copy()
source_label = "base scenario (script 1)"

if "e_id" in final_edges.columns:
    final_edges["e_id"] = final_edges["e_id"].astype(str)

# ---- pick flow column ----
if "acc_flow" in final_edges.columns:
    flow_col = "acc_flow"
elif "flow" in final_edges.columns:
    flow_col = "flow"
elif "current_flow" in final_edges.columns:
    flow_col = "current_flow"
else:
    raise ValueError("No usable flow column found. Expected one of: acc_flow, flow, current_flow")

# ---- build denominator (reference capacity) ----
if "current_capacity" in final_edges.columns:
    cap_ref = pd.to_numeric(final_edges["current_capacity"], errors="coerce")
    cap_source = "current_capacity"
elif {"acc_capacity", "acc_flow"}.issubset(final_edges.columns):
    # old-style outputs where acc_capacity is remaining capacity
    cap_ref = (
        pd.to_numeric(final_edges["acc_capacity"], errors="coerce")
        + pd.to_numeric(final_edges["acc_flow"], errors="coerce")
    )
    cap_source = "acc_flow + acc_capacity"
elif {"flow_capacity", "lanes"}.issubset(final_edges.columns):
    # heuristic fallback (often per-lane-per-hour)
    cap_ref = (
        pd.to_numeric(final_edges["flow_capacity"], errors="coerce")
        * pd.to_numeric(final_edges["lanes"], errors="coerce")
        * 24.0
    )
    cap_source = "flow_capacity * lanes * 24"
else:
    raise ValueError(
        "No usable capacity reference found. Expected current_capacity, "
        "or acc_capacity+acc_flow, or flow_capacity+lanes."
    )

flow_vals = pd.to_numeric(final_edges[flow_col], errors="coerce")
final_edges["vc_ratio"] = np.where(cap_ref > 0, flow_vals / cap_ref, np.nan)
final_edges["vc_ratio"] = pd.to_numeric(final_edges["vc_ratio"], errors="coerce")

# Basic diagnostics
valid_vc = final_edges["vc_ratio"].replace([np.inf, -np.inf], np.nan).dropna()
if len(valid_vc) == 0:
    raise ValueError("V/C computation produced no valid values.")

final_edges["vc_class"] = pd.cut(
    final_edges["vc_ratio"],
    bins=[-np.inf, 0.8, 1.0, np.inf],
    labels=["under-capacity (<=0.8)", "near-capacity (0.8-1.0)", "over-capacity (>1.0)"],
)

# ---- plot: map + histogram ----
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left panel: spatial V/C map
if skip_map_layers:
    axes[0].text(0.5, 0.5, "V/C map omitted for large study area", ha="center", va="center")
    axes[0].set_axis_off()
elif "geometry" in final_edges.columns and final_edges.geometry.notna().any():
    plot_gdf = subset_links_for_map(final_edges.copy(), flow_col=flow_col)
    if plot_gdf.crs is None:
        plot_gdf = plot_gdf.set_crs("EPSG:4326")
    q99 = float(valid_vc.quantile(0.99))
    vmax = max(1.0, q99)
    plot_gdf.plot(
        ax=axes[0],
        column="vc_ratio",
        cmap="RdYlGn_r",
        linewidth=1.2,
        legend=True,
        vmin=0,
        vmax=vmax,
        missing_kwds={"color": "#d1d5db", "label": "No data"},
    )
    axes[0].set_title("Base scenario link V/C ratio map")
    axes[0].set_axis_off()
else:
    axes[0].text(0.5, 0.5, "No geometry available for map plot", ha="center", va="center")
    axes[0].set_axis_off()

# Right panel: distribution + threshold lines
axes[1].hist(valid_vc, bins=40, color="#4c78a8", edgecolor="white", alpha=0.9)
axes[1].axvline(0.8, color="#f59e0b", linestyle="--", linewidth=1.4, label="V/C = 0.8")
axes[1].axvline(1.0, color="#ef4444", linestyle="--", linewidth=1.4, label="V/C = 1.0")
axes[1].set_title("Base scenario link V/C distribution")
axes[1].set_xlabel("V/C ratio")
axes[1].set_ylabel("Count of links")
axes[1].grid(True, axis="y", alpha=0.2)
axes[1].legend(frameon=False)

fig.suptitle(
    f"Step 4B | Base scenario V/C by link ({source_label}) | cap ref: {cap_source}",
    fontsize=13,
)
fig.tight_layout()
save_figure(fig, step4_fig_dir, "step4_base_vc_ratio")
plt.show()

# ---- concise numeric summary ----
print(f"V/C source file: {final_edge_path}")
print(f"Flow column used: {flow_col}")
print(f"Capacity reference: {cap_source}")
print(
    "V/C summary:",
    {
        "n_links": int(len(final_edges)),
        "n_valid_vc": int(valid_vc.size),
        "mean_vc": float(valid_vc.mean()),
        "median_vc": float(valid_vc.median()),
        "p95_vc": float(valid_vc.quantile(0.95)),
        "max_vc": float(valid_vc.max()),
        "over_capacity_share_pct": float((valid_vc > 1.0).mean() * 100.0),
    },
)

if "e_id" in final_edges.columns:
    top_cols = ["e_id", flow_col, "vc_ratio"]
    extra_cols = [c for c in ["current_capacity", "acc_capacity", "lanes"] if c in final_edges.columns]
    top = (
        final_edges[top_cols + extra_cols]
        .copy()
        .sort_values("vc_ratio", ascending=False)
        .head(10)
    )
    print("Top 10 links by V/C:")
    print(top.to_string(index=False, float_format=lambda v: f"{v:,.3f}"))


V/C source file: C:\Users\akothaw\Desktop\data\results\base_scenario\revision\edge_flows.gpq
Flow column used: acc_flow
Capacity reference: current_capacity
V/C summary: {'n_links': 483442, 'n_valid_vc': 483442, 'mean_vc': 4.867797837771661e-05, 'median_vc': 0.0, 'p95_vc': 0.00018870330158838043, 'max_vc': 0.008049650457401013, 'over_capacity_share_pct': 0.0}
Top 10 links by V/C:
   e_id  acc_flow  vc_ratio  current_capacity  acc_capacity  lanes
1101825   289.787     0.008             36000    35,710.213      1
 318224   265.974     0.007             36000    35,734.026      1
1621006   260.717     0.007             36000    35,739.283      1
 347397   357.266     0.006             60000    59,642.734      1
 347399   357.264     0.006             60000    59,642.736      1
 381998   343.621     0.006             60000    59,656.379      1
 382002   343.621     0.006             60000    59,656.379      1
 382013   343.611     0.006             60000    59,656.389      1
 351026   340.

## Optional future hazard map overlay

Use this section later when you want raster hazard layers and exposed assets on one map.
It follows your preferred style with log scaled flood depth and custom colormap.

In [22]:
# Optional: only run this when raster and map dependencies are installed and data paths are set.
#
# from matplotlib.lines import Line2D
# import matplotlib.colors as mcolors
# import cartopy.crs as ccrs
# import cartopy.feature as cfeature
# import rasterio
# from mpl_toolkits.axes_grid1.inset_locator import inset_axes
# from rasterio.enums import Resampling
# from rasterio.warp import transform_bounds
#
# EXTENT = [-120, -75, 23, 50]
# flood_cmap = mcolors.LinearSegmentedColormap.from_list(
#     "flood_purple", ["#deebf7","#dadaeb","#9e9ac8","#6a51a3","#3f007d"], N=256
# )
# flood_cmap.set_bad(alpha=0)
# flood_norm = mcolors.LogNorm(vmin=0.1, vmax=30)
#
# Fill in your hazard and exposure paths, then adapt from your existing map code.
print("Hazard map section is ready to customize when hazard layers are available.")

Hazard map section is ready to customize when hazard layers are available.


## Step 5. Export current figures + 100x (2x flood) visualization bundle

This section does **not** run pipeline scripts.
It only reads existing outputs and saves:
- a copy/export bundle of currently generated notebook figures
- comparison visuals for available flood scenarios under `od_inflation_100x` (expected: baseline/low/high IDs such as 1/2/3).

In [23]:
import shutil

export_root = fig_root / "exports"
current_bundle_dir = export_root / "100x_current_visualization"
comparison_bundle_dir = export_root / "100x_2x_flood"
current_bundle_dir.mkdir(parents=True, exist_ok=True)
comparison_bundle_dir.mkdir(parents=True, exist_ok=True)


def _copy_existing_figures(src_dir: Path, dst_dir: Path) -> int:
    dst_dir.mkdir(parents=True, exist_ok=True)
    copied = 0
    if not src_dir.exists():
        return copied
    for ext in ("*.png", "*.pdf"):
        for f in src_dir.glob(ext):
            shutil.copy2(f, dst_dir / f.name)
            copied += 1
    return copied


copied_total = 0
copied_total += _copy_existing_figures(step1_fig_dir, current_bundle_dir / "step1_base")
copied_total += _copy_existing_figures(step2_fig_dir, current_bundle_dir / "step2_disruption")
copied_total += _copy_existing_figures(step3_fig_dir, current_bundle_dir / "step3_damage")
copied_total += _copy_existing_figures(step4_fig_dir, current_bundle_dir / "step4_rerouting")

links_dir = results_root / "disruption_analysis" / VARIANT / str(DEPTH_KEY) / "links"
flood_ids = []
if links_dir.exists():
    for p in links_dir.glob("road_links_*.gpq"):
        tail = p.stem.split("_")[-1]
        if tail.isdigit():
            flood_ids.append(int(tail))
flood_ids = sorted(set(flood_ids))

rows = []
for fid in flood_ids:
    links_path = results_root / "disruption_analysis" / VARIANT / str(DEPTH_KEY) / "links" / f"road_links_{fid}.gpq"
    dmg_path = results_root / "damage_analysis" / VARIANT / f"intersections_{fid}_with_damage_values.csv"
    rer_path = results_root / "rerouting_analysis" / VARIANT / str(DEPTH_KEY) / str(fid) / "cost_matrix_by_scenario.csv"

    if not links_path.exists():
        continue

    links = gpd.read_parquet(links_path)
    flood_col_local = "flood_depth_max" if "flood_depth_max" in links.columns else next((c for c in links.columns if c.startswith("flood_depth")), None)
    depth_vals = pd.to_numeric(links[flood_col_local], errors="coerce") if flood_col_local else pd.Series(np.nan, index=links.index)
    exposed = depth_vals.fillna(0) > 0

    total_damage = np.nan
    if dmg_path.exists():
        dmg = prepare_damage_for_viz(pd.read_csv(dmg_path))
        total_damage = float(event_damage_total_usd(dmg))

    rerouting_s0 = np.nan
    combined_s0 = np.nan
    if rer_path.exists():
        rer = pd.read_csv(rer_path)
        if "scenario" in rer.columns:
            rer0 = rer.loc[pd.to_numeric(rer["scenario"], errors="coerce") == 0]
            if len(rer0) > 0:
                rerouting_s0 = float(pd.to_numeric(rer0.iloc[0].get("rerouting_cost", np.nan), errors="coerce"))
                combined_s0 = float(pd.to_numeric(rer0.iloc[0].get("combined_total_cost", np.nan), errors="coerce"))

    rows.append({
        "flood_id": fid,
        "n_links": int(len(links)),
        "exposed_links": int(exposed.sum()),
        "mean_depth_m": float(depth_vals[exposed].mean()) if int(exposed.sum()) > 0 else 0.0,
        "max_depth_m": float(depth_vals.max()) if depth_vals.notna().any() else 0.0,
        "direct_damage_usd": total_damage,
        "rerouting_cost_s0": rerouting_s0,
        "combined_total_s0": combined_s0,
    })

summary_2x = pd.DataFrame(rows).sort_values("flood_id").reset_index(drop=True)
summary_csv = comparison_bundle_dir / "summary_100x_2x_flood.csv"
summary_2x.to_csv(summary_csv, index=False)

# Figure A: depth profile comparison
if len(summary_2x) > 0:
    fig_a, ax_a = plt.subplots(figsize=(9, 5))
    x = summary_2x["flood_id"].astype(int).astype(str)
    ax_a.bar(x, summary_2x["mean_depth_m"], label="Mean exposed depth (m)", color="#4c78a8")
    ax_a.plot(x, summary_2x["max_depth_m"], marker="o", color="#e45756", linewidth=1.8, label="Max depth (m)")
    ax_a.set_title(f"100x OD | flood scenario depth profile (depth key={DEPTH_KEY})")
    ax_a.set_xlabel("Flood scenario ID")
    ax_a.set_ylabel("Depth (m)")
    ax_a.grid(True, axis="y", alpha=0.2)
    ax_a.legend(frameon=False)
    fig_a.tight_layout()
    save_figure(fig_a, comparison_bundle_dir, "depth_profile_100x_2x_flood")
    plt.show()

    # Figure B: cost comparison (direct + rerouting at scenario 0)
    fig_b, axes_b = plt.subplots(1, 2, figsize=(14, 5))

    axes_b[0].bar(x, summary_2x["direct_damage_usd"].fillna(0.0), color="#f58518")
    axes_b[0].set_title("Direct damage by flood scenario")
    axes_b[0].set_xlabel("Flood scenario ID")
    axes_b[0].set_ylabel("USD (consolidated direct damage)")
    axes_b[0].grid(True, axis="y", alpha=0.2)

    axes_b[1].bar(x, summary_2x["rerouting_cost_s0"].fillna(0.0), color="#72b7b2", label="Rerouting (S0)")
    axes_b[1].plot(x, summary_2x["combined_total_s0"].fillna(0.0), marker="o", color="#54a24b", label="Combined total (S0)")
    axes_b[1].axhline(0, color="black", linewidth=0.8)
    axes_b[1].set_title("Rerouting + combined total at scenario 0")
    axes_b[1].set_xlabel("Flood scenario ID")
    axes_b[1].set_ylabel("USD")
    axes_b[1].grid(True, axis="y", alpha=0.2)
    axes_b[1].legend(frameon=False)

    fig_b.tight_layout()
    save_figure(fig_b, comparison_bundle_dir, "cost_profile_100x_2x_flood")
    plt.show()

print(f"Copied existing figure files: {copied_total}")
print(f"Current visualization bundle: {current_bundle_dir}")
print(f"2x flood comparison bundle: {comparison_bundle_dir}")
print(f"Summary table: {summary_csv}")
print("Detected flood IDs:", flood_ids)
if len(summary_2x) > 0:
    print(summary_2x.to_string(index=False, float_format=lambda v: f"{v:,.3f}"))
else:
    print("No flood scenario files were detected under disruption links for this variant.")

Copied existing figure files: 29
Current visualization bundle: C:\Users\akothaw\Desktop\data\results\figures\revision\exports\100x_current_visualization
2x flood comparison bundle: C:\Users\akothaw\Desktop\data\results\figures\revision\exports\100x_2x_flood
Summary table: C:\Users\akothaw\Desktop\data\results\figures\revision\exports\100x_2x_flood\summary_100x_2x_flood.csv
Detected flood IDs: [1, 2, 3]
 flood_id  n_links  exposed_links  mean_depth_m  max_depth_m  direct_damage_usd  rerouting_cost_s0  combined_total_s0
        1   483599            124         0.772        0.809              2.735                NaN                NaN
        2   483599            124         0.541        0.567              2.110                NaN                NaN
        3   483599            124         0.927        0.971              3.149                NaN                NaN


In [24]:
# Consolidate ALL visualization pipeline plots into the 100x_2x_flood bundle
bundle_dir = fig_root / "exports" / "100x_2x_flood"
bundle_dir.mkdir(parents=True, exist_ok=True)

sources = {
    "step1_base": step1_fig_dir,
    "step2_disruption": step2_fig_dir,
    "step3_damage": step3_fig_dir,
    "step4_rerouting": step4_fig_dir,
}

copied_rows = []
total_copied = 0
for label, src in sources.items():
    dst = bundle_dir / label
    dst.mkdir(parents=True, exist_ok=True)
    if not src.exists():
        continue
    for ext in ("*.png", "*.pdf"):
        for f in src.glob(ext):
            out = dst / f.name
            shutil.copy2(f, out)
            copied_rows.append({
                "group": label,
                "source_path": str(f),
                "bundle_path": str(out),
                "ext": f.suffix.lower(),
            })
            total_copied += 1

# Also keep Step 5 comparison plots in root of 100x_2x_flood for convenience
for stem in ["depth_profile_100x_2x_flood", "cost_profile_100x_2x_flood"]:
    for ext in [".png", ".pdf"]:
        p = bundle_dir / f"{stem}{ext}"
        if p.exists():
            copied_rows.append({
                "group": "step5_comparison",
                "source_path": str(p),
                "bundle_path": str(p),
                "ext": ext,
            })

manifest = pd.DataFrame(copied_rows)
manifest_path = bundle_dir / "figure_manifest_100x_2x_flood.csv"
manifest.to_csv(manifest_path, index=False)

print(f"100x_2x_flood bundle path: {bundle_dir}")
print(f"Pipeline plots copied into bundle subfolders: {total_copied}")
print(f"Manifest: {manifest_path}")
if len(manifest) > 0:
    print(manifest.groupby("group").size().to_string())
else:
    print("No figures were found to bundle.")

100x_2x_flood bundle path: C:\Users\akothaw\Desktop\data\results\figures\revision\exports\100x_2x_flood
Pipeline plots copied into bundle subfolders: 29
Manifest: C:\Users\akothaw\Desktop\data\results\figures\revision\exports\100x_2x_flood\figure_manifest_100x_2x_flood.csv
group
step1_base           3
step2_disruption     6
step3_damage        12
step4_rerouting      8
step5_comparison     4


### Step 2A. Flood disruption scenario comparison (baseline, low, high)
This panel uses **risk-oriented metrics** that are easier to interpret:
- **Exposed length (km)**: total road length with flood depth > 0
- **Flow at risk (vehicles/day)**: baseline `acc_flow` allocated to exposed links via `e_id` join
- **Deep-flood share (%)**: percent of exposed links with depth > 0.5 m

Scenario sourcing logic remains the same:
- `baseline`: synthetic no-flood reference
- `low`: lowest available flood scenario ID
- `high`: highest distinct available flood scenario ID (fallback to `revision` variant if needed)

In [25]:
# Scenario comparison for flood disruption context (risk-oriented metrics)
# Always show Baseline / Low / High as three bars.

PRIMARY_VARIANT = VARIANT
FALLBACK_VARIANT = "revision"  # script-2 outputs often exist here

def _pick_flood_col(df: pd.DataFrame) -> str | None:
    if "flood_depth_max" in df.columns:
        return "flood_depth_max"
    return next((c for c in df.columns if c.startswith("flood_depth")), None)

def _pick_length_km(df: pd.DataFrame) -> pd.Series:
    if "length_mile" in df.columns:
        return pd.to_numeric(df["length_mile"], errors="coerce").fillna(0.0) * 1.60934
    if "length" in df.columns:
        # Commonly in meters for network links
        return pd.to_numeric(df["length"], errors="coerce").fillna(0.0) / 1000.0
    return pd.Series(0.0, index=df.index)

def _available_ids_for_variant(variant_name: str) -> list[int]:
    links_dir = results_root / "disruption_analysis" / variant_name / str(DEPTH_KEY) / "links"
    if not links_dir.exists():
        return []
    ids = []
    for p in links_dir.glob("road_links_*.gpq"):
        stem_tail = p.stem.split("_")[-1]
        if stem_tail.isdigit():
            ids.append(int(stem_tail))
    return sorted(set(ids))

# Baseline edge flow lookup for flow-at-risk metric

edge_flow_lookup = edge_flows[["e_id"]].copy()
edge_flow_lookup["e_id"] = edge_flow_lookup["e_id"].astype(str)
if "acc_flow" in edge_flows.columns:
    edge_flow_lookup["acc_flow_base"] = pd.to_numeric(edge_flows["acc_flow"], errors="coerce").fillna(0.0)
elif "flow" in edge_flows.columns:
    edge_flow_lookup["acc_flow_base"] = pd.to_numeric(edge_flows["flow"], errors="coerce").fillna(0.0)
elif "Car21" in edge_flows.columns:
    edge_flow_lookup["acc_flow_base"] = pd.to_numeric(edge_flows["Car21"], errors="coerce").fillna(0.0)
else:
    edge_flow_lookup["acc_flow_base"] = 0.0
edge_flow_lookup = edge_flow_lookup.drop_duplicates(subset=["e_id"])

def _scenario_stats(variant_name: str, scenario_id: int) -> dict:
    links_path = (
        results_root
        / "disruption_analysis"
        / variant_name
        / str(DEPTH_KEY)
        / "links"
        / f"road_links_{scenario_id}.gpq"
    )
    if not links_path.exists():
        return {
            "status": "missing_file_zeroed",
            "total_links": int(len(edge_flows)),
            "exposed_links": 0,
            "exposed_length_km": 0.0,
            "flow_at_risk": 0.0,
            "mean_exposed_depth_m": 0.0,
            "max_depth_m": 0.0,
            "deep_flood_share_pct": 0.0,
        }

    scen_links = gpd.read_parquet(links_path).copy()
    scen_links["e_id"] = scen_links["e_id"].astype(str)
    total_links = int(len(scen_links))

    flood_col_s = _pick_flood_col(scen_links)
    depth = pd.to_numeric(scen_links[flood_col_s], errors="coerce") if flood_col_s else pd.Series(np.nan, index=scen_links.index)
    exposed_mask = depth.fillna(0) > 0
    exposed_links = int(exposed_mask.sum())
    mean_exposed_depth_m = float(depth[exposed_mask].mean()) if exposed_links > 0 else 0.0
    max_depth_m = float(depth.max()) if depth.notna().any() else 0.0

    length_km = _pick_length_km(scen_links)
    exposed_length_km = float(length_km[exposed_mask].sum())

    scen_with_flow = scen_links.merge(edge_flow_lookup, on="e_id", how="left")
    flow_at_risk = float(pd.to_numeric(scen_with_flow.loc[exposed_mask, "acc_flow_base"], errors="coerce").fillna(0.0).sum())

    deep_flood_mask = depth.fillna(0) > 0.5
    deep_flood_share_pct = float((deep_flood_mask & exposed_mask).sum() / exposed_links * 100.0) if exposed_links > 0 else 0.0

    return {
        "status": "ok",
        "total_links": total_links,
        "exposed_links": exposed_links,
        "exposed_length_km": exposed_length_km,
        "flow_at_risk": flow_at_risk,
        "mean_exposed_depth_m": mean_exposed_depth_m,
        "max_depth_m": max_depth_m,
        "deep_flood_share_pct": deep_flood_share_pct,
    }

# Discover IDs from script-2 outputs
primary_ids = _available_ids_for_variant(PRIMARY_VARIANT)
fallback_ids = _available_ids_for_variant(FALLBACK_VARIANT)

# Pick Low/High with distinct scenario IDs when possible
if len(primary_ids) >= 2:
    low_id, high_id = min(primary_ids), max(primary_ids)
    low_variant, high_variant = PRIMARY_VARIANT, PRIMARY_VARIANT
elif len(primary_ids) == 1:
    low_id = primary_ids[0]
    low_variant = PRIMARY_VARIANT
    fallback_distinct = [i for i in fallback_ids if i != low_id]
    if fallback_distinct:
        high_id = max(fallback_distinct)
        high_variant = FALLBACK_VARIANT
    elif fallback_ids:
        high_id = max(fallback_ids)
        high_variant = FALLBACK_VARIANT
    else:
        high_id = low_id
        high_variant = PRIMARY_VARIANT
elif len(fallback_ids) >= 2:
    low_id, high_id = min(fallback_ids), max(fallback_ids)
    low_variant, high_variant = FALLBACK_VARIANT, FALLBACK_VARIANT
elif len(fallback_ids) == 1:
    low_id = high_id = fallback_ids[0]
    low_variant = high_variant = FALLBACK_VARIANT
else:
    low_id = high_id = 1
    low_variant = high_variant = PRIMARY_VARIANT

summary_rows = []
summary_rows.append(
    {
        "scenario_label": "baseline",
        "scenario_id": -1,
        "source_variant": PRIMARY_VARIANT,
        "status": "synthetic_baseline",
        "total_links": int(len(edge_flows)),
        "exposed_links": 0,
        "exposed_length_km": 0.0,
        "flow_at_risk": 0.0,
        "mean_exposed_depth_m": 0.0,
        "max_depth_m": 0.0,
        "deep_flood_share_pct": 0.0,
    }
)

low_stats = _scenario_stats(low_variant, low_id)
summary_rows.append({
    "scenario_label": "low",
    "scenario_id": int(low_id),
    "source_variant": low_variant,
    **low_stats,
})

high_stats = _scenario_stats(high_variant, high_id)
summary_rows.append({
    "scenario_label": "high",
    "scenario_id": int(high_id),
    "source_variant": high_variant,
    **high_stats,
})

summary_df = pd.DataFrame(summary_rows)
summary_df["scenario_label"] = pd.Categorical(summary_df["scenario_label"], ["baseline", "low", "high"], ordered=True)
summary_df = summary_df.sort_values("scenario_label").reset_index(drop=True)

# Risk-oriented metric charts with guaranteed 3 bars
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
labels = ["Baseline", "Low", "High"]
x = np.arange(len(labels))
# Use the centralized scenario palette
bar_colors = [c for _, _, c in SCENARIO_SPECS]

vals_length_km = summary_df["exposed_length_km"].fillna(0).to_numpy(dtype=float)
vals_flow_risk = summary_df["flow_at_risk"].fillna(0).to_numpy(dtype=float)
vals_deep_share = summary_df["deep_flood_share_pct"].fillna(0).to_numpy(dtype=float)

b0 = axes[0].bar(x, vals_length_km, color=bar_colors)
axes[0].set_title("Exposed network length")
axes[0].set_ylabel("Length (km)")
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels)
axes[0].grid(True, axis="y", alpha=0.2)
axes[0].set_ylim(0, max(vals_length_km.max(), 0.1) * 1.15)

b1 = axes[1].bar(x, vals_flow_risk, color=bar_colors)
axes[1].set_title("Flow at risk on exposed links")
axes[1].set_ylabel("Vehicles per day")
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels)
axes[1].grid(True, axis="y", alpha=0.2)
axes[1].set_ylim(0, max(vals_flow_risk.max(), 1.0) * 1.15)

b2 = axes[2].bar(x, vals_deep_share, color=bar_colors)
axes[2].set_title("Deep-flood share of exposed links")
axes[2].set_ylabel("Share (%)")
axes[2].set_xticks(x)
axes[2].set_xticklabels(labels)
axes[2].grid(True, axis="y", alpha=0.2)
axes[2].set_ylim(0, max(vals_deep_share.max(), 1.0) * 1.15)

for rect in b0:
    h = float(rect.get_height())
    axes[0].text(rect.get_x() + rect.get_width() / 2, h + axes[0].get_ylim()[1] * 0.02, f"{h:,.1f}", ha="center", va="bottom", fontsize=8)

for rect in b1:
    h = float(rect.get_height())
    axes[1].text(rect.get_x() + rect.get_width() / 2, h + axes[1].get_ylim()[1] * 0.02, f"{h:,.0f}", ha="center", va="bottom", fontsize=8)

for rect in b2:
    h = float(rect.get_height())
    axes[2].text(rect.get_x() + rect.get_width() / 2, h + axes[2].get_ylim()[1] * 0.02, f"{h:.1f}%", ha="center", va="bottom", fontsize=8)

# annotate scenario source (id + variant)
for i, row in summary_df.iterrows():
    sid = "baseline" if row["scenario_label"] == "baseline" else f"id={int(row['scenario_id'])}"
    axes[0].text(i, -axes[0].get_ylim()[1] * 0.10, f"{sid}\n{row['source_variant']}", ha="center", va="top", fontsize=8)

fig.suptitle(
    f"Flood disruption by scenario (risk metrics) | variant={PRIMARY_VARIANT} | depth key={DEPTH_KEY}",
    fontsize=12,
 )
fig.subplots_adjust(top=0.83, bottom=0.25, wspace=0.28)
save_figure(fig, step2_fig_dir, "step2_scenario_comparison_risk_metrics")

print("Flood disruption scenario summary (risk metrics)")
print(
    summary_df[[
        "scenario_label",
        "scenario_id",
        "source_variant",
        "status",
        "exposed_links",
        "exposed_length_km",
        "flow_at_risk",
        "mean_exposed_depth_m",
        "deep_flood_share_pct",
    ]].to_string(index=False, float_format=lambda v: f"{v:,.3f}")
)

plt.show()

Flood disruption scenario summary (risk metrics)
scenario_label  scenario_id source_variant             status  exposed_links  exposed_length_km  flow_at_risk  mean_exposed_depth_m  deep_flood_share_pct
      baseline           -1       revision synthetic_baseline              0              0.000         0.000                 0.000                 0.000
           low            1       revision                 ok            124             25.420         9.155                 0.772                97.581
          high            3       revision                 ok            124             25.420         9.155                 0.927                98.387


### Step 2A. Flood disruption scenario comparison (baseline, low, high)
This panel uses **risk-oriented metrics** that are easier to interpret:
- **Exposed length (km)**: total road length with flood depth > 0
- **Flow at risk (vehicles/day)**: baseline `acc_flow` allocated to exposed links via `e_id` join
- **Deep-flood share (%)**: percent of exposed links with depth > 0.5 m

Scenario sourcing logic remains the same:
- `baseline`: synthetic no-flood reference
- `low`: lowest available flood scenario ID
- `high`: highest distinct available flood scenario ID (fallback to `revision` variant if needed)

## Checkpoint: Final optimization choices (Script 2 + Script 4)

This section pins the final choices and visualizes before/after profile comparisons.

### Final choices
- **Script 2**: Candidate **C** (`NIRD_ENABLE_SPLIT_CACHE=1`)  
  Rationale: largest measured improvement with no output-change observed in prior validations.
- **Script 4**: cached batch path parsing (`NIRD_VECTORIZE_PATH_PARSING=1`)  
  Rationale: measurable reduction in `main()` profile time with low risk.

### Input profile summaries used
- `profiles/summary/2_intersection_analysis_depth30_event1_candidate_baseline_current_top.csv`
- `profiles/summary/2_intersection_analysis_depth30_event1_candidate_C_cache_top.csv`
- `profiles/summary/4_rerouting_candidate_baseline_final_top.csv`
- `profiles/summary/4_rerouting_candidate_AB_final_top.csv`

In [26]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

summary_dir = repo_root / "profiles" / "summary"
profile_inputs = [
    summary_dir / "2_intersection_analysis_depth30_event1_candidate_baseline_current_top.csv",
    summary_dir / "2_intersection_analysis_depth30_event1_candidate_C_cache_top.csv",
    summary_dir / "4_rerouting_candidate_baseline_final_top.csv",
    summary_dir / "4_rerouting_candidate_AB_final_top.csv",
]

if not all(p.exists() for p in profile_inputs):
    print("Skipping profile comparison plot; profile summaries are not present.")
else:
    s2_base_csv, s2_opt_csv, s4_base_csv, s4_opt_csv = profile_inputs

    def metric_from_csv(path: Path, func_contains: str | None = None) -> float:
        df = pd.read_csv(path)
        if func_contains is None:
            return float(df.iloc[0]["cumtime"])
        marker = df["func"].str.contains(func_contains, regex=False)
        if not marker.any():
            raise ValueError(f"Function marker '{func_contains}' not found in {path.name}")
        return float(df.loc[marker, "cumtime"].iloc[0])

    s2_base = metric_from_csv(s2_base_csv)
    s2_opt = metric_from_csv(s2_opt_csv)
    s4_base = metric_from_csv(s4_base_csv, "4_rerouting_and_recovery_scenario_loop.py:255(main)")
    s4_opt = metric_from_csv(s4_opt_csv, "4_rerouting_and_recovery_scenario_loop.py:255(main)")

    plot_df = pd.DataFrame(
        {
            "Script": ["Script 2", "Script 2", "Script 4", "Script 4"],
            "Variant": ["Baseline", "Optimized (C)", "Baseline", "Optimized (path cache)"],
            "Cumtime_s": [s2_base, s2_opt, s4_base, s4_opt],
        }
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=120)
    s2 = plot_df[plot_df["Script"] == "Script 2"]
    axes[0].bar(s2["Variant"], s2["Cumtime_s"], color=["#6c757d", "#2ca02c"])
    axes[0].set_title("Script 2: Baseline vs Candidate C")
    axes[0].set_ylabel("Cumulative time (s)")
    axes[0].tick_params(axis="x", rotation=15)

    s4 = plot_df[plot_df["Script"] == "Script 4"]
    axes[1].bar(s4["Variant"], s4["Cumtime_s"], color=["#6c757d", "#1f77b4"])
    axes[1].set_title("Script 4: Baseline vs Optimized")
    axes[1].set_ylabel("Cumulative time (s)")
    axes[1].tick_params(axis="x", rotation=15)

    for ax, base, opt in [(axes[0], s2_base, s2_opt), (axes[1], s4_base, s4_opt)]:
        pct = (base - opt) / base * 100.0 if base else 0.0
        ax.text(0.5, max(base, opt) * 0.95, f"Improvement: {pct:.1f}%", ha="center", va="top", fontsize=10)

    plt.tight_layout()
    plt.show()

    print("Script 2 baseline:", round(s2_base, 2), "s")
    print("Script 2 optimized:", round(s2_opt, 2), "s")
    print("Script 4 baseline main():", round(s4_base, 2), "s")
    print("Script 4 optimized main():", round(s4_opt, 2), "s")


Skipping profile comparison plot; profile summaries are not present.


### Notes and interpretation

- Script 2 shows the largest gain from split-cache reuse (Candidate C), so this is the recommended production configuration.
- Script 4 optimization reduces parsing overhead by caching repeated path-string decoding and batch-mapping unique payloads.
- For Script 4 comparison, `main()` cumulative time is preferred over module-level total because import/runtime environment overhead can dominate total top-row cumtime.

This checkpoint is also documented in `PERFORMANCE_CHECKPOINT_2026-05-06.md`.